<a href="https://colab.research.google.com/github/Tqhuyen/glaucoma-thesis/blob/master/notebooks/3d_glaucoma_multiview_sota_sweep_xai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multiview SOTA Backbone Sweep + Fusion Ablation + X-AI (glaucoma, Harvard-GF 200^3)

Upgrade of `3d_glaucoma_multiview_2d3d.ipynb`. Same multi-stream idea (1 x 3D volume branch +
3 x 2D en-face "report-like" views), but the small from-scratch encoders are replaced by a **registry of
modern backbones**, the 4 fusion ops are extended with **cross-attention+gate / Mamba-SSM / FiLM-SE**,
every run logs to **wandb**, and the **best model is explained with X-AI** (3D & 2D Grad-CAM, occlusion
sensitivity, integrated gradients, fusion attention, LIME-at-view level). All metrics (acc / balanced-acc /
precision / recall / specificity / F1 / MCC / AUC / PR-AUC / ECE + bootstrap CI) + figures are saved locally
**and** to Drive under `MasterBKDN/Thesis/`.

## Dataset
`harvardairobotics/Harvard-GF`: 3,300 OCT volumes, 200^3 uint8, per-scan `.npz['oct_bscans']`.
Consolidated arrays are built (once, cached) at raw **200^3** (`STORE_RES=200`, never downsampled on disk).
Resolution policy = **keep the original 200^3 whenever possible**: all conv/from-scratch/MONAI-conv 3D
backbones run at raw 200 (no resampling). Only patch/window transformers (SwinUNETR, UNETR, ViT-3D)
drop to 160-192 because their patch/window grid requires divisibility (and VRAM); each such row carries
its own `res3d` in the sweep table. The 3 en-face views (`aip_full`, `slab_aip`, `slab_mip`) are
projected once and cached.

## Honest runtime expectation (Colab A100 80GB, bf16, resume-safe)
Every experiment row is saved when it finishes and its figures (ROC/PR/calibration, confusion, val
history) are written to Drive + logged to wandb immediately - you see each result as it completes.
- Tier A "single 3D branch": conv @ raw 200^3 ~ 10-25 min/run; transformer rows @160 ~ 10-30 min/run.
- Tier B "single 2D view (pretrained timm)": fast, ~2-6 min/run after ImageNet weights download.
- Tier C "multiview fusion": conv-3D @ raw 200 + 3x 2D backbones x fusion ops.
- Tier D X-AI on the winner: minutes.
Budget roughly **3-9 h** depending on how many tiers/rows you enable. Reduce `CFG["epochs"]`, disable
transformer rows, or lower a row's `res3d` to shrink it. `MVX_SMOKE=1` validates everything on CPU
(synthetic data) before you spend GPU hours.


In [1]:
import os, sys, io, json, time, math, shutil, zipfile, csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    pass

def _pip(pkgs):
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

_pip(["monai", "timm", "scikit-learn", "scikit-image", "scipy", "matplotlib", "pandas", "shap"])
_pip(["hf-transfer", "huggingface_hub", "datasets", "wandb"])

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("WANDB_SILENT", "true")

# --- 3DINO-ViT (gated) deps + repo for TIER A "single3d-3dino" ---
_pip(["omegaconf", "fvcore", "iopath", "torchmetrics"])
import subprocess as _sp3d
if not os.path.isdir("/content/3DINO"):
    _sp3d.run(["git", "clone", "--depth", "1", "https://github.com/AICONSlab/3DINO.git", "/content/3DINO"])
if "/content/3DINO" not in sys.path:
    sys.path.insert(0, "/content/3DINO")
print("[3dino] repo ready:", os.path.isdir("/content/3DINO"))


'true'

In [2]:
try:
    from google.colab import drive, userdata, output
except Exception:
    drive = userdata = output = None
if drive is not None:
    drive.mount("/content/drive")
    for k in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            if userdata.get(k):
                os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
    print("[drive] mounted OK")
    JS = """
setInterval(function(){
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.click();
}, 60000);
"""
    try:
        output.eval_js(JS)
        print("[keepalive] armed")
    except Exception as e:
        print("[keepalive] n/a:", e)
else:
    print("[env] not running in Colab (Drive mount skipped). Set WANDB_MODE/HF_TOKEN manually if needed.")


Mounted at /content/drive
[drive] mounted OK
[keepalive] armed


In [3]:
SMOKE = os.environ.get("MVX_SMOKE", "0") == "1"
RUN_SWEEP = os.environ.get("MVX_SWEEP", "1") == "1"
RUN_XAI = os.environ.get("MVX_XAI", "1") == "1"
os.environ.setdefault("WANDB_MODE", "offline" if SMOKE else os.environ.get("WANDB_MODE", "online"))
os.environ.setdefault("WANDB_API_KEY", "local")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
BF16 = bool(USE_AMP and torch.cuda.is_bf16_supported())
AMP_DT = torch.bfloat16 if BF16 else torch.float16
SEED = 42

STORE_RES = 200
MODEL_RES3D_DEFAULT = 192
RES2D = 224
VIEWS = ["aip_full", "slab_aip", "slab_mip"]

LOCAL_ROOT = "/content" if not os.name == "nt" else os.path.join(os.path.expanduser("~"), "mvx_work")
DRIVE_ROOT = "/content/drive/MyDrive/MasterBKDN/Thesis"
EXPERIMENT = "multiview_sota_sweep"
SAVE_DIR = os.path.join(LOCAL_ROOT, EXPERIMENT)
DRIVE_DIR = os.path.join(DRIVE_ROOT, EXPERIMENT)

print("smoke=", SMOKE, "sweep=", RUN_SWEEP, "xai=", RUN_XAI, "device=", DEVICE,
      "amp=", AMP_DT if USE_AMP else "none", "colab=", IN_COLAB)


smoke= False sweep= True xai= True device= cuda amp= torch.bfloat16 colab= True


## 1. Configuration

`DATA_MODE` decides where volumes come from:
- `hf`: stream `harvardairobotics/Harvard-GF` zip (used by the older sweep notebooks),
- `dir`: read already-consolidated `{split}_volumes.npy` from `DATA_DIR`.

Views + depth-axis are **precomputed once per split and cached**, so every subsequent epoch /
experiment only reads the 3 small view arrays + memmaps the 3D volume (resized per backbone).


In [4]:
CFG = {
    "num_classes": 2,
    "data_mode": "hf",                 # "hf" | "dir"
    "hf_repo": "harvardairobotics/Harvard-GF",
    "hf_zip": "Dataset/dataset.zip",
    "hf_csv": "ReadMe/data_summary.csv",
    "data_dir": "/content/glaucoma_hf_200",
    "epochs": 15,
    "batch_size": 2,
    "grad_accum": 8,
    "lr": 2e-4,
    "wd": 1e-4,
    "patience": 6,
    "log_every": 40,
    "num_workers": 0,
    "latent": 256,
    "max_train": 0,
    "xai_samples_per_class": 2,
    "force_xai": False,
}
SPLITS = ("Training", "Validation", "Test")
SPLIT_ALIAS = {"training": "Training", "validation": "Validation", "valid": "Validation",
               "test": "Test", "testing": "Test", "testing2": "Test"}

if SMOKE:
    CFG.update({
        "epochs": 1, "batch_size": 1, "grad_accum": 1, "log_every": 5,
        "latent": 32, "num_workers": 0, "xai_samples_per_class": 1,
        "smoke_sz": 48, "n_smoke_train": 6, "n_smoke_val": 3, "n_smoke_test": 2,
    })

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True) if os.path.exists(DRIVE_ROOT) else None

def set_seed(s=SEED):
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def jdump(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2)

def jload(path, default=None):
    try:
        with open(path) as fh:
            return json.load(fh)
    except Exception:
        return default

def copy_to_drive(path, sub=""):
    if not os.path.exists(DRIVE_ROOT):
        return path
    dst_dir = os.path.join(DRIVE_DIR, sub)
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, os.path.basename(path))
    try:
        shutil.copy(path, dst)
    except Exception as e:
        print("[drive] copy fail", path, e)
    return dst

def init_wandb(run_name, config=None):
    if os.environ.get("WANDB_MODE") != "offline" and not os.environ.get("WANDB_API_KEY"):
        print("[wandb] key missing; continue without cloud")
        return None
    try:
        import wandb
        run = wandb.init(project="glaucoma-thesis", name=run_name,
                         config=config or {}, id=run_name, resume="allow")
        return run
    except Exception as e:
        print("[wandb] init failed:", e)
        return None

def wandb_log(wb, d):
    if wb is not None:
        try:
            wb.log(d)
        except Exception as e:
            print("[wandb] log fail", e)

def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6


## 2. Data: build consolidated 200^3 + cached views + datasets/loaders

Two entry points:
- `build_data()` -> consolidated `{Training,Validation,Test}_{volumes,labels}.npy` at raw 200^3
  (idempotent, reused across runs / notebooks), plus a `manifest.json`.
- `ensure_views()` -> for every split computes and caches `{split}_views.npy` (N,3,200,200) uint8,
  `{split}_dzs.npy` (N,) and `{split}_labels.npy`. Projections follow the reference notebook:
  depth-axis auto-detected per volume, RNFL slab = bright peak +/- `slab_half`, then
  `aip_full` (mean over depth), `slab_aip` (mean over slab), `slab_mip` (max over slab).


In [5]:
DATA_DIR = CFG["data_dir"]

def hf_download(filename):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=CFG["hf_repo"], filename=filename, repo_type="dataset")

def read_csv_meta(csv_path):
    meta = {}
    with open(csv_path, newline="") as fh:
        for r in csv.DictReader(fh):
            split = SPLIT_ALIAS.get((r["use"] or "").strip().lower())
            if split is None:
                continue
            gl = 1 if str(r["glaucoma"]).strip().lower() in ("yes", "1", "true") else 0
            meta[os.path.splitext(os.path.basename(r["filename"]))[0]] = (split, gl)
    return meta

def zip_npz_names(zip_path):
    import zipfile
    with zipfile.ZipFile(zip_path) as zf:
        return sorted(n for n in zf.namelist() if n.endswith(".npz"))

def build_data_from_hf():
    if all(os.path.isfile(os.path.join(DATA_DIR, f"{s}_volumes.npy")) for s in SPLITS):
        print("[data] consolidated arrays already present in", DATA_DIR)
        return
    os.makedirs(DATA_DIR, exist_ok=True)
    meta = read_csv_meta(hf_download(CFG["hf_csv"]))
    zip_path = hf_download(CFG["hf_zip"])
    names = zip_npz_names(zip_path)
    names = [n for n in names if os.path.splitext(os.path.basename(n))[0] in meta]
    counts = {}
    for s in SPLITS:
        counts[s] = sum(1 for n in names if meta[os.path.splitext(os.path.basename(n))[0]][0] == s)
    print("[data] matched counts:", counts)
    vols, labs = {}, {}
    for s in SPLITS:
        vp = os.path.join(DATA_DIR, f"{s}_volumes.npy")
        vols[s] = np.lib.format.open_memmap(vp, mode="w+", dtype=np.uint8,
                                            shape=(counts[s], 1, STORE_RES, STORE_RES, STORE_RES))
        labs[s] = np.zeros(counts[s], dtype=np.int64)
    filled = {s: 0 for s in SPLITS}
    with zipfile.ZipFile(zip_path) as zf:
        for n in names:
            m = meta[os.path.splitext(os.path.basename(n))[0]]
            split, label = m
            raw = np.load(io.BytesIO(zf.read(n)))["oct_bscans"]
            assert raw.shape == (STORE_RES,) * 3 and raw.dtype == np.uint8, (n, raw.shape, raw.dtype)
            vols[split][filled[split]] = raw[None]
            labs[split][filled[split]] = label
            filled[split] += 1
    for s in SPLITS:
        vols[s].flush()
        np.save(os.path.join(DATA_DIR, f"{s}_labels.npy"), labs[s])
        print(f"[data] {s}: {filled[s]} vols")
    jdump({"source": CFG["hf_repo"], "store_res": STORE_RES, "splits": {s: filled[s] for s in SPLITS}},
          os.path.join(DATA_DIR, "manifest.json"))

def ensure_local_dir_data():
    for s in SPLITS:
        for f in (f"{s}_volumes.npy", f"{s}_labels.npy"):
            assert os.path.isfile(os.path.join(DATA_DIR, f)), f"missing {f} in {DATA_DIR}"

def build_data():
    if CFG["data_mode"] == "hf":
        build_data_from_hf()
    else:
        ensure_local_dir_data()

def counts():
    return {s: int(np.load(os.path.join(DATA_DIR, f"{s}_labels.npy")).shape[0]) for s in SPLITS}


In [6]:
def depth_axis(vol):
    f = vol.astype(np.float32)
    stds = [float(f.mean(axis=tuple(i for i in range(3) if i != ax)).std()) for ax in range(3)]
    return int(np.argmax(stds))

def to_depth_last(vol, dz):
    if dz == 2:
        return vol
    others = [i for i in range(3) if i != dz]
    return np.transpose(vol, tuple(others) + (dz,))

def project_views(dvol, methods=VIEWS, half=16):
    S = dvol.shape[2]
    half = min(int(half), max(2, S // 8))
    prof = dvol.mean(axis=(0, 1)).astype(np.float32)
    peak = int(prof.argmax())
    lo, hi = max(0, peak - half), min(S, peak + half + 1)
    out = []
    for m in methods:
        if m == "aip_full":
            out.append(dvol.mean(axis=2))
        elif m == "slab_aip":
            out.append(dvol[:, :, lo:hi].mean(axis=2))
        elif m == "slab_mip":
            out.append(dvol[:, :, lo:hi].max(axis=2))
        else:
            raise ValueError(m)
    return np.stack([np.round(o).clip(0, 255).astype(np.uint8) for o in out], axis=0)

def ensure_views():
    build_data()
    for s in SPLITS:
        vp = os.path.join(DATA_DIR, f"{s}_views.npy")
        dzp = os.path.join(DATA_DIR, f"{s}_dzs.npy")
        if os.path.isfile(vp) and os.path.isfile(dzp):
            continue
        n = counts()[s]
        vols = np.load(os.path.join(DATA_DIR, f"{s}_volumes.npy"), mmap_mode="r")
        views = np.lib.format.open_memmap(vp, mode="w+", dtype=np.uint8, shape=(n, len(VIEWS), STORE_RES, STORE_RES))
        dzs = np.zeros(n, dtype=np.int8)
        for i in range(n):
            raw = np.ascontiguousarray(vols[i][0])
            dz = depth_axis(raw)
            dzs[i] = dz
            views[i] = project_views(to_depth_last(raw, dz))
            if (i + 1) % 300 == 0:
                views.flush()
                print(f"[views] {s} {i + 1}/{n}")
        views.flush()
        np.save(dzp, dzs)
        print(f"[views] done {s} n={n}")

def counts_splits():
    return counts()


`MVRealDS`: reads the raw 200^3 volume from disk, resizes on-the-fly to `res3d` (only when !=200)
and to `res2d` for the 3 cached views, normalizes to `[0,1]`. Each item: `x` (1,res3d^3), `v`
(3,res2d,res2d), `labels`, `idx`. `MVSynthDS` (SMOKE) builds random volumes with a bright slab.


In [7]:
class MVRealDS(Dataset):
    def __init__(self, split, res3d=MODEL_RES3D_DEFAULT, res2d=RES2D, half=16,
                 load3d=True, loadviews=True):
        self.split, self.res3d, self.res2d, self.half = split, int(res3d), int(res2d), int(half)
        self.load3d, self.loadviews = bool(load3d), bool(loadviews)
        self.labels = np.load(os.path.join(DATA_DIR, f"{split}_labels.npy"))
        self.volumes = np.load(os.path.join(DATA_DIR, f"{split}_volumes.npy"), mmap_mode="r") if load3d else None
        self.views = np.load(os.path.join(DATA_DIR, f"{split}_views.npy"), mmap_mode="r") if loadviews else None
        self.dzs = np.load(os.path.join(DATA_DIR, f"{split}_dzs.npy")) if load3d else None
        print(f"[ds] {split} n={len(self.labels)} res3d={res3d} res2d={res2d} load3d={load3d} loadviews={loadviews}")

    def __len__(self):
        return len(self.labels)

    def raw_volume(self, i):
        return np.ascontiguousarray(self.volumes[i][0])

    def __getitem__(self, i):
        if self.load3d:
            raw = self.raw_volume(i)
            d = int(self.dzs[i])
            if d != 2:
                raw = to_depth_last(raw, d)
            x = torch.from_numpy(raw[None].astype(np.float32) / 255.0)
            if self.res3d != STORE_RES:
                x = F.interpolate(x.unsqueeze(0), size=(self.res3d,) * 3, mode="trilinear", align_corners=False).squeeze(0)
        else:
            x = torch.zeros(1, self.res3d, self.res3d, self.res3d)
        if self.loadviews:
            v = torch.from_numpy(self.views[i].astype(np.float32) / 255.0)
            if self.res2d != STORE_RES:
                v = F.interpolate(v.unsqueeze(0), size=(self.res2d,) * 2, mode="bilinear", align_corners=False).squeeze(0)
        else:
            v = torch.zeros(len(VIEWS), self.res2d, self.res2d)
        y = torch.tensor(int(self.labels[i]), dtype=torch.long)
        return {"x": x, "v": v, "labels": y, "idx": torch.tensor(int(i), dtype=torch.long)}

class MVSynthDS(Dataset):
    def __init__(self, n, res3d=48, res2d=64, half=6):
        self.res3d, self.res2d, self.half = res3d, res2d, half
        self.items = []
        rng = np.random.default_rng(0)
        for _ in range(n):
            S = res3d
            vol = rng.integers(0, 15, size=(S, S, S), dtype=np.uint8)
            r0, r1 = S // 4, min(S - 1, S // 4 + max(4, S // 5))
            vol[:, :, r0:r1] = np.clip(vol[:, :, r0:r1].astype(np.int16) + 45, 0, 255).astype(np.uint8)
            lab = int(rng.integers(0, 2))
            c, h = S // 2, max(3, S // 8)
            if lab:
                vol[c - h:c + h, c - h:c + h, r0:r1] = np.clip(
                    vol[c - h:c + h, c - h:c + h, r0:r1].astype(np.int16) + 90, 0, 255).astype(np.uint8)
            views, _ = project_views(vol, VIEWS, half)
            x = torch.from_numpy(vol[None].astype(np.float32) / 255.0)
            v = F.interpolate(torch.from_numpy(views[None].astype(np.float32) / 255.0),
                              size=(res2d,) * 2, mode="bilinear", align_corners=False).squeeze(0)
            self.items.append({"x": x, "v": v, "labels": torch.tensor(lab, dtype=torch.long), "idx": 0})

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

def collate_dict(batch):
    out = {k: torch.stack([b[k] for b in batch]) for k in ("x", "v", "labels")}
    out["idx"] = torch.tensor([int(b["idx"]) for b in batch], dtype=torch.long)
    return out

def make_loader(ds, split_kind, bs, workers, shuffle):
    if split_kind != "train":
        workers = 0
    kw = dict(batch_size=bs, num_workers=workers, collate_fn=collate_dict,
              pin_memory=(DEVICE.type == "cuda"), drop_last=False, shuffle=shuffle)
    if workers > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    if split_kind == "train":
        g = torch.Generator()
        g.manual_seed(SEED)
        kw["generator"] = g
    return DataLoader(ds, **kw)

def build_loaders(res3d=MODEL_RES3D_DEFAULT, res2d=RES2D, bs=None, max_train=0,
                  load3d=True, loadviews=True):
    def effective_workers():
        if os.name == "nt" or DEVICE.type != "cuda":
            return 0
        return int(CFG.get("num_workers", 2) or 0)
    workers = effective_workers()
    bs = CFG["batch_size"] if bs is None else bs
    ensure_views()
    train_ds = MVRealDS("Training", res3d, res2d, load3d=load3d, loadviews=loadviews)
    val_ds = MVRealDS("Validation", res3d, res2d, load3d=load3d, loadviews=loadviews)
    test_ds = MVRealDS("Test", res3d, res2d, load3d=load3d, loadviews=loadviews)
    if max_train and max_train < len(train_ds):
        g = torch.Generator().manual_seed(SEED)
        keep = torch.randperm(len(train_ds), generator=g)[:max_train].tolist()
        train_ds = torch.utils.data.Subset(train_ds, keep)
    return (make_loader(train_ds, "train", bs, workers, True),
            make_loader(val_ds, "val", bs, workers, False),
            make_loader(test_ds, "test", bs, workers, False))

def build_smoke_loaders():
    workers = 0
    c = CFG
    tr = DataLoader(MVSynthDS(c["n_smoke_train"], 48, 64, 6), batch_size=c["batch_size"], collate_fn=collate_dict)
    va = DataLoader(MVSynthDS(c["n_smoke_val"], 48, 64, 6), batch_size=c["batch_size"], collate_fn=collate_dict)
    te = DataLoader(MVSynthDS(c["n_smoke_test"], 48, 64, 6), batch_size=c["batch_size"], collate_fn=collate_dict)
    return tr, va, te


## 3. Branch encoders (registry)

3D branch options (return a pooled embedding `out_dim`, input `(B,1,D,H,W)` in `[0,1]`):
- `convnext3d` — ConvNeXt-style 3D (depthwise conv + LayerNorm-on-channels + MLP), from scratch,
  anisotropic pooling `(2,2,1)` on early stages to protect the thin RNFL slab along depth.
- `resxt3d` — grouped-bottleneck 3D residual (ResNeXt-style) from scratch.
- `cnn3d` — reference micro-ResNet-3D from `multiview_2d3d` (kept for ablation).
- `vit3d` — patch-embed ViT-3D (patch 16) from scratch (needs res divisible by 16).
- MONAI `vnet` / `segresnet` / `dynunet` / `unetr` / `swinunetr`, nnU-Net v2 encoders and MONAI 3D
  ResNet `mednet10/18/50` (optional MedicalNet pretrained best-effort) -> next cell.

2D branch options (each en-face view -> its own encoder, `(B,1,H,W) -> out_dim`):
- `tiny2d` — from-scratch ResNet-like, no ImageNet (ablation baseline).
- timm pretrained: `convnextv2_tiny`, `convnext_tiny`, `swin_tiny_patch4_window7_224`,
  `vit_base_patch16_224`, `deit3_small_patch16_224`, `maxvit_tiny_rw_224`. Views resized to 224.


In [8]:
def find_last_conv(m, conv_cls=nn.Conv3d):
    last = None
    for name, mod in m.named_modules():
        if isinstance(mod, conv_cls):
            last = mod
    return last

def gnorm(c, groups=None):
    g = min(c, 8) if groups is None else groups
    while c % g != 0:
        g -= 1
    return nn.GroupNorm(g, c)

class LN3d(nn.Module):
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(c, eps)
    def forward(self, x):
        x = x.permute(0, 2, 3, 4, 1).contiguous()
        x = self.norm(x)
        return x.permute(0, 4, 1, 2, 3).contiguous()

class CNBlock3D(nn.Module):
    def __init__(self, c, ks=3, expand=4):
        super().__init__()
        self.norm = LN3d(c)
        self.dw = nn.Conv3d(c, c, ks, padding=ks // 2, groups=c)
        self.pw1 = nn.Conv3d(c, c * expand, 1)
        self.gelu = nn.GELU()
        self.pw2 = nn.Conv3d(c * expand, c, 1)
        self.ls = nn.Parameter(torch.ones(c, 1, 1, 1))
    def forward(self, x):
        h = self.pw2(self.gelu(self.pw1(self.dw(self.norm(x)))))
        return x + h * self.ls

class Down3D(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv = nn.Conv3d(cin, cout, 3, stride=stride, padding=1)
        self.norm = LN3d(cout)
    def forward(self, x):
        return self.norm(self.conv(x))

class Enc3DConvNeXt(nn.Module):
    def __init__(self, in_ch=1, features=(24, 48, 96, 192), depths=(2, 2, 4, 2),
                 strides=((2, 2, 1), (2, 2, 2), (2, 2, 2), (2, 2, 2)), gcam_stage=2):
        super().__init__()
        self.out_dim = int(features[-1])
        self.gcam_stage = gcam_stage
        self.stem = nn.Conv3d(in_ch, features[0], 3, padding=1)
        self.stages = nn.ModuleList()
        cin = features[0]
        for i, (c, n, st) in enumerate(zip(features, depths, strides)):
            blocks = []
            if i > 0:
                blocks.append(Down3D(cin, c, _pad_stride(st)))
            blocks += [CNBlock3D(c) for _ in range(n)]
            self.stages.append(nn.Sequential(*blocks))
            cin = c
        self.pool = nn.AdaptiveAvgPool3d(1)
    def forward(self, x):
        x = self.stem(x)
        for st in self.stages:
            x = st(x)
        return self.pool(x).flatten(1)
    def gcam_module(self):
        return find_last_conv(self.stages[self.gcam_stage])

def _pad_stride(s):
    return tuple(int(x) for x in s)

class ResXBlock3D(nn.Module):
    def __init__(self, cin, cout, stride=(1, 1, 1), groups=8, base_width=8):
        super().__init__()
        width = int(cout * base_width / 64.0) * groups
        self.c1 = nn.Conv3d(cin, width, 1, bias=False)
        self.g1 = gnorm(width)
        self.c2 = nn.Conv3d(width, width, 3, stride=stride, padding=1, groups=groups, bias=False)
        self.g2 = gnorm(width)
        self.c3 = nn.Conv3d(width, cout, 1, bias=False)
        self.g3 = gnorm(cout)
        self.short = (stride != (1, 1, 1) or cin != cout)
        if self.short:
            self.sc = nn.Conv3d(cin, cout, 1, stride=stride, bias=False)
            self.sg = gnorm(cout)
    def forward(self, x):
        r = self.c3(F.relu(self.g2(self.c2(F.relu(self.g1(self.c1(x)))))))
        if self.short:
            x = self.sg(self.sc(x))
        return F.relu(x + r)

class Enc3DResNeXt(nn.Module):
    def __init__(self, in_ch=1, features=(32, 64, 128, 192), strides=((2, 2, 1), (2, 2, 2), (2, 2, 2), (2, 2, 2))):
        super().__init__()
        self.out_dim = int(features[-1])
        self.stem = nn.Sequential(nn.Conv3d(in_ch, features[0], 3, padding=1, bias=False), gnorm(features[0]), nn.ReLU())
        self.stages = nn.ModuleList()
        cin = features[0]
        for i, (c, st) in enumerate(zip(features, strides)):
            if i > 0:
                self.stages.append(ResXBlock3D(cin, c, _pad_stride(st)))
                cin = c
            self.stages.append(ResXBlock3D(cin, c, (1, 1, 1)))
        self.pool = nn.AdaptiveAvgPool3d(1)
    def forward(self, x):
        x = self.stem(x)
        for st in self.stages:
            x = st(x)
        return self.pool(x).flatten(1)

class ResBlock3D(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv3d(cin, cout, 3, stride=stride, padding=1)
        self.g1 = gnorm(cout)
        self.c2 = nn.Conv3d(cout, cout, 3, padding=1)
        self.g2 = gnorm(cout)
        self.short = (stride != (1, 1, 1) or cin != cout)
        if self.short:
            self.sc = nn.Conv3d(cin, cout, 1, stride=stride)
            self.sg = gnorm(cout)
    def forward(self, x):
        r = self.g2(self.c2(F.relu(self.g1(self.c1(x)))))
        if self.short:
            x = self.sg(self.sc(x))
        return F.relu(x + r)

class Enc3DCNN(nn.Module):
    def __init__(self, in_ch=1, features=(24, 48, 96, 192), depth_strides=(1, 1, 1, 2), gcam_stage=3):
        super().__init__()
        self.out_dim = int(features[-1])
        self.gcam_stage = gcam_stage
        c0 = features[0]
        self.stem = nn.Sequential(nn.Conv3d(in_ch, c0, 3, padding=1), gnorm(c0), nn.ReLU())
        blocks = []
        for i, cout in enumerate(features):
            cin = features[i - 1] if i > 0 else c0
            lat = 1 if i == 0 else 2
            blocks.append(ResBlock3D(cin, cout, (lat, lat, depth_strides[i])))
        self.blocks = nn.ModuleList(blocks)
        self.pool = nn.AdaptiveAvgPool3d(1)
    def forward(self, x):
        x = self.stem(x)
        for b in self.blocks:
            x = b(x)
        return self.pool(x).flatten(1)
    def gcam_module(self):
        return find_last_conv(self.blocks[self.gcam_stage])

class PatchEmbed3D(nn.Module):
    def __init__(self, in_ch, dim, patch):
        super().__init__()
        self.patch = patch
        self.proj = nn.Conv3d(in_ch, dim, patch, stride=patch)
    def forward(self, x):
        t = self.proj(x)
        b, c = t.shape[:2]
        return t.flatten(2).transpose(1, 2)

class Enc3DViT(nn.Module):
    def __init__(self, in_ch=1, patch=16, dim=192, depth=6, nhead=6, mlp=768, res=192):
        super().__init__()
        self.patch = patch
        self.out_dim = dim
        self.embed = PatchEmbed3D(in_ch, dim, patch)
        npos = (res // patch) ** 3
        self.pos = nn.Parameter(torch.zeros(1, npos, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=nhead, dim_feedforward=mlp,
                                           dropout=0.1, batch_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(dim)
    def _npos(self, x):
        d = x.shape[2] // self.patch
        return d ** 3
    def forward(self, x):
        t = self.embed(x)
        if t.shape[1] != self.pos.shape[1]:
            p = torch.zeros(1, t.shape[1], self.pos.shape[2], device=t.device)
            nn.init.trunc_normal_(p, std=0.02)
            self.pos = nn.Parameter(p)
        t = t + self.pos[:, : t.shape[1]]
        t = self.enc(t)
        return self.norm(t).mean(1)
    def gcam_module(self):
        return self.embed.proj


MONAI segmentation backbones are used *encoder-only* (decoder dropped) + a projection to the shared
fusion space, exactly like `seg_backbones_sweep`. UNETR / SwinUNETR need the volume divisible by 16/32
(res3d override). MONAI / nnU-Net import failures degrade gracefully (row skipped with a message).


In [9]:
import monai
from monai.networks import nets as _monai_nets

class SegEncHead(nn.Module):
    def __init__(self, backbone, num_classes=2, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.out_dim = backbone.out_dim
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(backbone.out_dim, num_classes))
    def forward(self, x):
        return self.head(self.backbone(x))

class MONAIDynUNetEnc(nn.Module):
    def __init__(self, in_channels=1, filters=(16, 32, 64, 128, 256), res_block=True):
        super().__init__()
        self.net = _monai_nets.DynUNet(
            spatial_dims=3, in_channels=in_channels, out_channels=1,
            kernel_size=[3] * len(filters), strides=[1] + [2] * (len(filters) - 1),
            upsample_kernel_size=[2] * len(filters), filters=list(filters),
            norm_name="batch", act_name=("relu", {"inplace": True}), res_block=res_block)
        self.out_dim = filters[-1]
    def forward(self, x):
        x = self.net.input_block(x)
        for d in self.net.downsamples:
            x = d(x)
        return F.adaptive_avg_pool3d(self.net.bottleneck(x), 1).flatten(1)

class MONAIVNetEnc(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.net = _monai_nets.VNet(spatial_dims=3, in_channels=in_channels, out_channels=1)
        self.out_dim = 256
    def forward(self, x):
        o16 = self.net.in_tr(x)
        o32 = self.net.down_tr32(o16)
        o64 = self.net.down_tr64(o32)
        o128 = self.net.down_tr128(o64)
        o256 = self.net.down_tr256(o128)
        return F.adaptive_avg_pool3d(o256, 1).flatten(1)

class MONAISegResEnc(nn.Module):
    def __init__(self, in_channels=1, init_filters=16):
        super().__init__()
        self.net = _monai_nets.SegResNet(spatial_dims=3, in_channels=in_channels, out_channels=1,
                                         init_filters=init_filters)
        self.out_dim = init_filters * 2 ** (len(self.net.blocks_down) - 1)
    def forward(self, x):
        x, _ = self.net.encode(x)
        return F.adaptive_avg_pool3d(x, 1).flatten(1)

class MONAIUNETREnc(nn.Module):
    def __init__(self, in_channels=1, hidden_size=192, num_heads=6, mlp_dim=768,
                 img_size=(160, 160, 160)):
        super().__init__()
        self.img_size = tuple(int(i) for i in img_size)
        self.net = _monai_nets.UNETR(in_channels=in_channels, out_channels=1, img_size=self.img_size,
                                     feature_size=16, hidden_size=hidden_size, mlp_dim=mlp_dim,
                                     num_heads=num_heads, proj_type="conv", norm_name="instance")
        self.out_dim = hidden_size
    def forward(self, x):
        tokens, _ = self.net.vit(x)
        return tokens.mean(dim=1)

class MONAISwinUNETREnc(nn.Module):
    def __init__(self, in_channels=1, feature_size=24, depths=(2, 2, 2, 2), num_heads=(3, 6, 12, 24)):
        super().__init__()
        self.net = _monai_nets.SwinUNETR(in_channels=in_channels, out_channels=1,
                                         feature_size=feature_size, depths=depths,
                                         num_heads=num_heads, window_size=7, spatial_dims=3)
        self.out_dim = feature_size * 2 ** len(depths)
    def forward(self, x):
        hs = self.net.swinViT(x, True)
        return F.adaptive_avg_pool3d(hs[-1], 1).flatten(1)

class Enc3DINO(nn.Module):
    def __init__(self, repo_dir="/content/3DINO", weights_repo="AICONSlab/3DINO-ViT",
                 cfg_name="train/vit3d_highres", out_dim=1024):
        super().__init__()
        if repo_dir not in sys.path:
            sys.path.insert(0, repo_dir)
        from huggingface_hub import hf_hub_download
        from dinov2.configs import load_and_merge_config_3d
        from dinov2.eval.setup import build_model_for_eval
        ckpt = hf_hub_download(repo_id=weights_repo, filename="3dino_vit_weights.pth")
        cfg = load_and_merge_config_3d(cfg_name)
        self.net = build_model_for_eval(cfg, ckpt)
        for p in self.net.parameters():
            p.requires_grad_(False)
        self.net.eval()
        self.out_dim = int(out_dim)

    @staticmethod
    def _prep(x):
        b = x.shape[0]
        xf = x.reshape(b, -1)
        lo = torch.quantile(xf, 0.0005, dim=1, keepdim=True)
        hi = torch.quantile(xf, 0.9995, dim=1, keepdim=True)
        shape = (b,) + (1,) * (x.ndim - 1)
        x = (x - lo.view(shape)) / (hi.view(shape) - lo.view(shape) + 1e-6)
        return torch.clip(x * 2 - 1, -1, 1)

    def forward(self, x):
        return self.net(self._prep(x))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


In [10]:
class BasicBlock3DMed(nn.Module):
    expansion = 1
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(cout)
        self.conv2 = nn.Conv3d(cout, cout, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv3d(cin, cout, 1, stride=stride, bias=False), nn.BatchNorm3d(cout))
    def forward(self, x):
        return F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))) + self.shortcut(x))

class Enc3DResNetMed(nn.Module):
    def __init__(self, in_channels=1, layers=(1, 1, 1, 1), inplanes=(64, 128, 256, 512)):
        super().__init__()
        self.out_dim = inplanes[-1]
        self.conv1 = nn.Conv3d(in_channels, inplanes[0], 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm3d(inplanes[0])
        self.pool = nn.MaxPool3d(3, stride=2, padding=1)
        self.stages = nn.ModuleList()
        cin = inplanes[0]
        for i, (c, n) in enumerate(zip(inplanes, layers)):
            seq = []
            for j in range(n):
                seq.append(BasicBlock3DMed(cin, c, stride=2 if (j == 0 and i > 0) else 1))
                cin = c
            self.stages.append(nn.Sequential(*seq))
        self.gap = nn.AdaptiveAvgPool3d(1)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        for st in self.stages:
            x = st(x)
        return self.gap(x).flatten(1)

MEDICALNET_URL = {
    "mednet10": "https://github.com/Tencent/MedicalNet/releases/download/v1.0/medicalnet_resnet10_23dsample.pth",
    "mednet18": "https://github.com/Tencent/MedicalNet/releases/download/v1.0/medicalnet_resnet18_23dsample.pth",
    "mednet50": "https://github.com/Tencent/MedicalNet/releases/download/v1.0/medicalnet_resnet50_23dsample.pth",
}

def _url_to_cache(url):
    name = os.path.basename(url)
    p = os.path.join(LOCAL_ROOT, "medicalnet_ckpts", name)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.isfile(p):
        print(f"[mednet] downloading {name} ...")
        import urllib.request
        import socket
        prev = socket.getdefaulttimeout()
        socket.setdefaulttimeout(90)
        try:
            urllib.request.urlretrieve(url, p)
        finally:
            socket.setdefaulttimeout(prev)
    return p

def medicalnet_pretrain(model, url):
    loaded = 0
    total = sum(p.numel() for p in model.parameters())
    try:
        sd = torch.load(_url_to_cache(url), map_location="cpu")
        if isinstance(sd, dict) and "state_dict" in sd:
            sd = sd["state_dict"]
        msd = model.state_dict()
        cand = {}
        for k, v in sd.items():
            kk = k.replace("module.", "")
            if kk in msd and msd[kk].shape == v.shape:
                cand[kk] = v
        model.load_state_dict(cand, strict=False)
        loaded = sum(v.numel() for v in cand.values())
        print(f"[mednet] transferred {loaded / 1e6:.1f}M/{total / 1e6:.1f}M params "
              f"({100.0 * loaded / max(total, 1):.0f}%)")
    except Exception as e:
        print(f"[mednet] pretrain unavailable ({type(e).__name__}: {str(e)[:160]}); using random init")
    return loaded / max(total, 1) > 0.5


In [11]:
def build3d(spec):
    kind = spec["kind"]
    params = spec.get("params", {})
    if kind == "convnext3d":
        return Enc3DConvNeXt(**params)
    if kind == "resxt3d":
        return Enc3DResNeXt(**params)
    if kind == "cnn3d":
        return Enc3DCNN(**params)
    if kind == "vit3d":
        return Enc3DViT(**params)
    if kind == "dynunet":
        return MONAIDynUNetEnc(**params)
    if kind == "vnet":
        return MONAIVNetEnc(**params)
    if kind == "segresnet":
        return MONAISegResEnc(**params)
    if kind == "unetr":
        return MONAIUNETREnc(**params)
    if kind == "swinunetr":
        return MONAISwinUNETREnc(**params)
    if kind.startswith("mednet"):
        layers = {"mednet10": (1, 1, 1, 1), "mednet18": (2, 2, 2, 2), "mednet50": (3, 4, 6, 3)}[kind]
        inplanes = tuple(params.get("inplanes", (64, 128, 256, 512)))
        m = Enc3DResNetMed(layers=layers, inplanes=inplanes)
        if params.get("pretrained") and kind in MEDICALNET_URL:
            medicalnet_pretrain(m, MEDICALNET_URL[kind])
        return m
    if kind == "3dino":
        return Enc3DINO(**params)
    raise ValueError(f"unknown 3d backbone {kind}")


In [12]:
class ResBlock2D(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.g1 = gnorm(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False)
        self.g2 = gnorm(cout)
        self.short = (stride != 1 or cin != cout)
        if self.short:
            self.sc = nn.Conv2d(cin, cout, 1, stride=stride, bias=False)
            self.sg = gnorm(cout)
    def forward(self, x):
        r = self.g2(self.c2(F.relu(self.g1(self.c1(x)))))
        if self.short:
            x = self.sg(self.sc(x))
        return F.relu(x + r)

class Enc2DCNN(nn.Module):
    def __init__(self, in_ch=1, features=(32, 64, 128, 256), blocks=(2, 2, 2, 2), gcam_stage=3):
        super().__init__()
        self.out_dim = int(features[-1])
        self.gcam_stage = gcam_stage
        c0 = features[0]
        self.stem = nn.Sequential(nn.Conv2d(in_ch, c0, 3, padding=1, bias=False), gnorm(c0), nn.ReLU())
        layers = []
        for i, (cout, nb) in enumerate(zip(features, blocks)):
            cin = features[i - 1] if i > 0 else c0
            for b in range(nb):
                layers.append(ResBlock2D(cin if b == 0 else cout, cout, stride=(2 if (i > 0 and b == 0) else 1)))
        self.body = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        x = self.stem(x)
        x = self.body(x)
        return self.pool(x).flatten(1)
    def gcam_module(self):
        return find_last_conv(self.body[self.gcam_stage], nn.Conv2d)

class TimmEnc(nn.Module):
    def __init__(self, name, pretrained=True):
        super().__init__()
        import timm
        self.name = name
        self.net = timm.create_model(name, pretrained=pretrained, num_classes=0, global_pool="avg")
        self.out_dim = self.net.num_features
    def forward(self, x):
        if x.ndim == 4 and x.shape[1] == 1:
            x = x.expand(-1, 3, -1, -1)
        return self.net(x)

def build2d(kind, name=None, pretrained=True, params=None):
    params = params or {}
    if kind == "tiny2d":
        return Enc2DCNN(**params)
    if kind == "timm":
        return TimmEnc(name or "convnextv2_tiny", pretrained=pretrained)
    raise ValueError(f"unknown 2d kind {kind}")

def list_timm_candidates():
    cands = ["convnextv2_tiny", "convnext_tiny", "swin_tiny_patch4_window7_224",
             "vit_base_patch16_224", "deit3_small_patch16_224", "maxvit_tiny_rw_224"]
    ok = []
    for n in cands:
        try:
            import timm
            if n in timm.list_models():
                ok.append(n)
        except Exception:
            break
    return ok


In [13]:
def _nheads(D):
    for h in (8, 4, 2, 1):
        if D % h == 0:
            return h
    return 1

class Proj(nn.Module):
    def __init__(self, d, D):
        super().__init__()
        self.fc = nn.Linear(d, D)
    def forward(self, x):
        return F.relu(self.fc(x))

class FusionAttn(nn.Module):
    def __init__(self, D, nb, heads=None):
        super().__init__()
        self.D, self.nb = D, nb
        h = heads or _nheads(D)
        self.norm1 = nn.LayerNorm(D)
        self.attn = nn.MultiheadAttention(D, h, dropout=0.1, batch_first=True)
        self.norm2 = nn.LayerNorm(D)
        self.ff = nn.Sequential(nn.Linear(D, 2 * D), nn.ReLU(), nn.Linear(2 * D, D))
        self.cls = nn.Parameter(torch.zeros(D))
        self.last_w = None
        self.last_cls_w = None
    def forward(self, toks):
        b, t, _ = toks.shape
        cls = self.cls.unsqueeze(0).unsqueeze(0).expand(b, -1, -1)
        x = torch.cat([cls, toks], dim=1)
        h = self.norm1(x)
        out, w = self.attn(h, h, h, need_weights=True)
        x = x + out
        x = x + self.ff(self.norm2(x))
        self.last_w = w.detach()
        self.last_cls_w = w[:, 0, 1:].detach().mean(0)
        return x[:, 0]

class FusionCrossGate(nn.Module):
    def __init__(self, D, nb, heads=None):
        super().__init__()
        h = heads or _nheads(D)
        self.norm = nn.LayerNorm(D)
        self.attn = nn.MultiheadAttention(D, h, dropout=0.1, batch_first=True)
        self.gate = nn.Parameter(torch.tensor(0.5))
        self.last_w = None
    def forward(self, c3, toks2d):
        q = c3.unsqueeze(1)
        h = self.norm(toks2d)
        out, w = self.attn(q, h, h, need_weights=True)
        self.last_w = w[..., 0, :].detach()
        z = c3 + torch.sigmoid(self.gate) * out.squeeze(1)
        return z

class DSSM(nn.Module):
    def __init__(self, din, d_state=16):
        super().__init__()
        self.din, self.d_state = din, d_state
        self.a_log = nn.Parameter(torch.randn(din, d_state) * 0.01)
        self.b = nn.Parameter(torch.randn(din, d_state) * 0.01)
        self.c = nn.Parameter(torch.randn(din, d_state) * 0.01)
        self.dt_log = nn.Parameter(torch.randn(din) * 0.01)
    def forward(self, u):
        b, t, din = u.shape
        a = -torch.exp(self.a_log).clamp(max=8.0)
        dt = F.softplus(self.dt_log).clamp(max=0.1)
        a_bar = torch.exp(a * dt[:, None])
        b_bar = (a_bar - 1.0) / (a + 1e-6) * self.b
        uu = u.transpose(0, 1)
        h = torch.zeros(b, din, self.d_state, device=u.device)
        outs = []
        for tt in range(t):
            h = h * a_bar.unsqueeze(0) + b_bar.unsqueeze(0) * uu[tt].unsqueeze(-1)
            outs.append((h * self.c.unsqueeze(0)).sum(-1))
        return torch.stack(outs, dim=1)

class FusionMamba(nn.Module):
    def __init__(self, D, nb):
        super().__init__()
        self.nb = nb
        self.in_proj = nn.Linear(D, 2 * D)
        self.down = nn.Linear(D, 64)
        self.ssm = DSSM(64, 16)
        self.up = nn.Linear(64, D)
        self.norm = nn.LayerNorm(D)
        self.head = nn.Linear(D, 2)
    def forward(self, toks):
        x = toks
        g = self.in_proj(x)
        z, u = g.chunk(2, dim=-1)
        z = F.silu(z)
        u = self.ssm(self.down(u))
        h = self.up(u)
        h = h * z
        y = self.norm(x + h)
        return self.head(y.mean(dim=1))

class FusionFiLM(nn.Module):
    def __init__(self, D, nb):
        super().__init__()
        self.nb = nb
        self.head = nn.Linear(D, 2)
        self.gate = nn.Sequential(nn.Linear(nb, nb), nn.ReLU(), nn.Linear(nb, nb), nn.Sigmoid())
    def forward(self, toks):
        mean = toks.mean(dim=2)
        w = self.gate(mean).unsqueeze(-1)
        return self.head((toks * w).sum(dim=1))


In [14]:
class MultiViewModel(nn.Module):
    def __init__(self, cfg, spec):
        super().__init__()
        self.cfg = cfg
        self.spec = spec
        self.mode = spec["fusion"]
        self.view_idx = spec.get("view")
        self.nv = len(VIEWS)
        D = int(cfg["latent"])
        nc = int(cfg["num_classes"])
        self.names = []
        enc3d_spec = spec.get("enc3d")
        enc2d_spec = spec.get("enc2d")
        if self.mode == "single2d":
            self.enc3d = None
            vi = self.view_idx
            self.enc2ds = nn.ModuleList([build2d(**enc2d_spec)])
            self.names = [f"view{vi}:{VIEWS[vi]}"]
            self.head = nn.Linear(self.enc2ds[0].out_dim, nc)
            return
        if enc3d_spec is not None:
            self.enc3d = build3d(enc3d_spec)
            self.names.append("enc3d")
        else:
            self.enc3d = None
        if self.mode == "single3d":
            self.enc2ds = nn.ModuleList()
            self.head = nn.Linear(self.enc3d.out_dim, nc)
            return
        self.enc2ds = nn.ModuleList([build2d(**enc2d_spec) for _ in range(self.nv)])
        for i in range(self.nv):
            self.names.append(f"view{i}:{VIEWS[i]}")
        d3 = self.enc3d.out_dim if self.enc3d is not None else D
        d2 = self.enc2ds[0].out_dim
        if self.mode == "single3d":
            return
        dims = []
        if self.enc3d is not None:
            dims.append(d3)
        dims += [d2] * self.nv
        self.projs = nn.ModuleList([Proj(d, D) for d in dims])
        nb = len(dims)
        if self.mode == "concat":
            self.head = nn.Linear(D * nb, nc)
        elif self.mode in ("add", "mul", "attn", "crossgate"):
            if self.mode == "attn":
                self.fusion = FusionAttn(D, nb)
            elif self.mode == "crossgate":
                self.fusion = FusionCrossGate(D, self.nv)
            self.head = nn.Linear(D, nc)
        elif self.mode == "mamba":
            self.fusion = FusionMamba(D, nb)
            self.head = None
        elif self.mode == "film":
            self.fusion = FusionFiLM(D, nb)
            self.head = None
        else:
            raise ValueError(self.mode)

    def embed(self, x, v):
        es = []
        if self.enc3d is not None:
            es.append(self.enc3d(x))
        es += [enc(v[:, i:i + 1]) for i, enc in enumerate(self.enc2ds)]
        return es

    def forward(self, x, v):
        if self.mode == "single2d":
            return self.head(self.enc2ds[0](v[:, self.view_idx:self.view_idx + 1]))
        if self.mode == "single3d":
            return self.head(self.enc3d(x))
        es = self.embed(x, v)
        toks = torch.stack([p(e) for p, e in zip(self.projs, es)], dim=1)
        if self.mode == "concat":
            return self.head(toks.reshape(toks.shape[0], -1))
        if self.mode == "add":
            return self.head(toks.sum(dim=1))
        if self.mode == "mul":
            return self.head(torch.prod(toks + 1e-6, dim=1))
        if self.mode == "attn":
            return self.head(self.fusion(toks))
        if self.mode == "crossgate":
            z = self.fusion(toks[:, 0], toks[:, 1:])
            return self.head(z)
        if self.mode == "mamba":
            return self.fusion(toks)
        if self.mode == "film":
            return self.fusion(toks)
        raise ValueError(self.mode)

    def enc_gcam(self, enc):
        if enc is None:
            return None
        if hasattr(enc, "gcam_module") and callable(getattr(enc, "gcam_module")):
            try:
                m = enc.gcam_module()
                if m is not None:
                    return m
            except Exception:
                pass
        return find_last_conv(enc) or find_last_conv(enc, nn.Conv2d)

    def gcam3d_module(self):
        return self.enc_gcam(self.enc3d)

    def gcam2d_module(self, i):
        return self.enc_gcam(self.enc2ds[i])

def build_model(cfg, spec):
    return MultiViewModel(cfg, spec)


## 4. Metrics (full clinical + calibration set)

All metrics computed per split for the positive (glaucoma) class: accuracy, balanced accuracy,
precision/PPV, recall/sensitivity, specificity, NPV, F1, MCC, ROC-AUC, PR-AUC, expected calibration
error (ECE). `metrics_full` returns a dict of scalars; `eval_metrics` wraps it around a trained model
and returns probabilities too (used for X-AI / curve plots).


In [15]:
def metrics_full(y_true, y_prob, thresh=0.5):
    from sklearn import metrics as skm
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    y_pred = (y_prob >= thresh).astype(int)
    acc = float(np.mean(y_true == y_pred))
    bacc = float(skm.balanced_accuracy_score(y_true, y_pred))
    tn, fp, fn, tp = skm.confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    def safe(d):
        return float(d) if d == d and abs(d) != np.inf else 0.0
    prec = safe(tp / max(tp + fp, 1))
    rec = safe(tp / max(tp + fn, 1))
    spec = safe(tn / max(tn + fp, 1))
    npv = safe(tn / max(tn + fn, 1))
    f1 = safe(2 * prec * rec / max(prec + rec, 1e-12))
    mcc = safe((tp * tn - fp * fn) / math.sqrt(max((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn), 1e-12)))
    both = len(np.unique(y_true)) > 1
    auc = safe(skm.roc_auc_score(y_true, y_prob)) if both else float("nan")
    pr_auc = safe(skm.average_precision_score(y_true, y_prob)) if both else float("nan")
    ece = ece_score(y_true, y_prob, n_bins=10)
    eps = 1e-12
    logloss = float(-np.mean(y_true * np.log(np.clip(y_prob, eps, 1 - eps))
                            + (1 - y_true) * np.log(np.clip(1 - y_prob, eps, 1 - eps))))
    brier = float(np.mean((y_prob - y_true) ** 2))
    n = len(y_true)
    po = (tp + tn) / max(n, 1)
    pe = ((tp + fn) * (tp + fp) + (tn + fp) * (tn + fn)) / max(n * n, 1)
    kappa = float((po - pe) / max(1 - pe, 1e-12)) if pe < 1 else float("nan")
    prec0 = safe(tn / max(tn + fn, 1))
    rec0 = safe(tn / max(tn + fp, 1))
    f0 = safe(2 * prec0 * rec0 / max(prec0 + rec0, 1e-12))
    f1_macro = float((f0 + f1) / 2)
    youden = float(rec + spec - 1)
    return {"acc": acc, "balanced_acc": bacc, "precision": prec, "recall": rec, "specificity": spec,
            "npv": npv, "f1": f1, "f1_macro": f1_macro, "mcc": mcc, "roc_auc": auc, "pr_auc": pr_auc,
            "ece": ece, "logloss": logloss, "brier": brier, "kappa": kappa, "youden": youden,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp), "n": int(n)}

def ece_score(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    y_true = np.asarray(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if hi == 1.0:
            m = (y_prob >= lo) & (y_prob <= hi)
        if m.sum() == 0:
            continue
        conf = float(y_prob[m].mean())
        acc = float(y_true[m].mean())
        ece += (m.sum() / len(y_true)) * abs(conf - acc)
    return float(ece)

def bootstrap_ci(y_true, y_prob, metric="roc_auc", n_boot=500, seed=SEED):
    from sklearn import metrics as skm
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    vals = []
    n = len(y_true)
    if len(np.unique(y_true)) < 2:
        return (float("nan"), float("nan"))
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        if metric == "roc_auc":
            vals.append(skm.roc_auc_score(yt, yp))
        elif metric == "pr_auc":
            vals.append(skm.average_precision_score(yt, yp))
        else:
            vals.append(float(np.mean((yp >= 0.5) == yt)))
    vals = np.asarray(vals)
    if len(vals) < 30:
        return (float("nan"), float("nan"))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))

def optimal_youden_threshold(y_val, p_val, grid=99):
    from sklearn import metrics as skm
    y_val = np.asarray(y_val)
    p_val = np.asarray(p_val, dtype=np.float64)
    best_t, best_j = 0.5, -1.0
    for t in np.linspace(0.0, 1.0, grid):
        yp = (p_val >= t).astype(int)
        tn, fp, fn, tp = skm.confusion_matrix(y_val, yp, labels=[0, 1]).ravel()
        sens = tp / max(tp + fn, 1)
        spec = tn / max(tn + fp, 1)
        j = sens + spec - 1
        if j > best_j:
            best_j, best_t = j, float(t)
    return best_t, float(best_j)

def metrics_at_threshold(y_true, y_prob, th):
    return metrics_full(y_true, y_prob, thresh=th)

@torch.no_grad()
def predict_proba(model, loader, desc=""):
    model.eval()
    ys, ps, losses = [], [], 0.0
    crit = nn.CrossEntropyLoss()
    for b in loader:
        x = b["x"].to(DEVICE)
        v = b["v"].to(DEVICE)
        y = b["labels"].to(DEVICE)
        if USE_AMP:
            with torch.autocast("cuda", dtype=AMP_DT):
                logits = model(x, v)
        else:
            logits = model(x, v)
        probs = torch.softmax(logits.float(), dim=1)[:, 1]
        ys.extend(y.cpu().numpy().tolist())
        ps.extend(probs.cpu().numpy().tolist())
        losses += crit(logits.float(), y).item() * y.numel()
    return np.asarray(ys), np.asarray(ps), losses / max(len(ys), 1)


In [16]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def plot_cm(y_true, y_pred, path, title=""):
    from sklearn import metrics as skm
    cm = skm.confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["no-glaucoma", "glaucoma"])
    ax.set_yticklabels(["no-glaucoma", "glaucoma"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
    if title:
        ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)
    return path

def plot_curves(y_true, y_prob, path):
    from sklearn import metrics as skm
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
    fpr, tpr, _ = skm.roc_curve(y_true, y_prob)
    axes[0].plot(fpr, tpr, label=f"AUC={skm.roc_auc_score(y_true, y_prob):.3f}")
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set_title("ROC"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend()
    pr, rc, _ = skm.precision_recall_curve(y_true, y_prob)
    axes[1].plot(rc, pr, label=f"AP={skm.average_precision_score(y_true, y_prob):.3f}")
    axes[1].set_title("Precision-Recall"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend()
    bin_conf, bin_acc = [], []
    edges = np.linspace(0, 1, 11)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (y_prob >= lo) & (y_prob <= hi)
        if m.sum():
            bin_conf.append(y_prob[m].mean()); bin_acc.append(y_true[m].mean())
    axes[2].plot([0, 1], [0, 1], "--", color="gray")
    axes[2].plot(bin_conf, bin_acc, "o-")
    axes[2].set_title("Calibration (ECE)").set_xlabel("Confidence"); axes[2].set_ylabel("Accuracy")
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)
    return path

def plot_metric_table(rows, path):
    keys = ["name", "params", "val_acc", "val_auc", "test_acc", "test_f1", "test_auc", "test_mcc", "status"]
    cols = {k: [r.get(k, "") for r in rows] for k in keys}
    fig, ax = plt.subplots(figsize=(0.42 * len(keys) + 4, 0.42 * len(rows) + 2.2))
    ax.axis("off")
    t = ax.table(cellText=[[cols[k][i] for k in keys] for i in range(len(rows))], colLabels=keys, loc="center")
    t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1.0, 1.25)
    fig.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return path

def fmt_metric_row(name, m):
    return {"name": name, "acc": round(m["acc"], 4), "balanced_acc": round(m["balanced_acc"], 4),
            "precision": round(m["precision"], 4), "recall": round(m["recall"], 4),
            "specificity": round(m["specificity"], 4), "npv": round(m["npv"], 4),
            "f1": round(m["f1"], 4), "f1_macro": round(m["f1_macro"], 4), "mcc": round(m["mcc"], 4),
            "roc_auc": round(m["roc_auc"], 4), "pr_auc": round(m["pr_auc"], 4),
            "ece": round(m["ece"], 4), "logloss": round(m["logloss"], 4), "brier": round(m["brier"], 4),
            "kappa": round(m["kappa"], 4), "youden": round(m["youden"], 4),
            "tn": m["tn"], "fp": m["fp"], "fn": m["fn"], "tp": m["tp"], "n": m["n"]}


## 5. Probe + training loop

`probe` checks a model fits one forward/backward at the given batch (auto-drops to batch 1 on OOM) and
reports params + peak VRAM. `train_one` mirrors the repo pipeline: AdamW (lr scaled by
`sqrt(effective_bs/4)`), cosine annealing, AMP (bf16 when available, fp16+GradScaler otherwise),
grad-accumulation, early stop on val, full test metrics on the best epoch, checkpoint + per-run JSON,
live wandb logging of the complete metric set. Every row is saved resume-safe.


In [17]:
def _acc_from_loader(model, loader):
    yt, yp, _ = predict_proba(model, loader)
    return float(np.mean((yp >= 0.5) == yt)) if len(yt) else float("nan")

def probe(model, loader, bs):
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    opt = torch.optim.AdamW(model.parameters(), 1e-4)
    try:
        b = next(iter(loader))
        x = b["x"].to(DEVICE); v = b["v"].to(DEVICE); y = b["labels"].to(DEVICE)
        if USE_AMP:
            with torch.autocast("cuda", dtype=AMP_DT):
                loss = F.cross_entropy(model(x, v), y)
        else:
            loss = F.cross_entropy(model(x, v), y)
        loss.backward(); opt.step()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        vram = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
        del model, opt, x, v, y, loss
        torch.cuda.empty_cache()
        return {"params": count_params(model) if hasattr(model, "out_dim") else 0.0,
                "vram": vram, "status": "ok", "bs": bs}
    except RuntimeError as e:
        torch.cuda.empty_cache()
        if bs > 1 and "out of memory" in str(e).lower():
            print("  [probe] OOM at bs", bs, "-> retry bs1")
            return probe(model, bs=1, loader=loader)
        st = "OOM" if "out of memory" in str(e).lower() else "ERR"
        return {"params": float("nan"), "vram": float("nan"), "status": st, "bs": bs}

def train_one(cfg, spec, loaders, run_dir, wb=None):
    set_seed()
    name = spec["name"]
    tr, va, te = loaders
    model = build_model(cfg, spec).to(DEVICE)
    npar = count_params(model)
    eff = cfg["batch_size"] * cfg["grad_accum"]
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"] * (eff / 4) ** 0.5, weight_decay=cfg["wd"])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and not BF16))
    crit = nn.CrossEntropyLoss()
    best_val, best_te, bad, step = -1.0, None, 0, 0
    history = []
    t0 = time.time()
    print(f"[{name}] params={npar:.2f}M lr={opt.param_groups[0]['lr']:.2e}")
    for ep in range(cfg["epochs"]):
        model.train()
        opt.zero_grad(set_to_none=True)
        run_loss = 0.0
        n_see = 0
        for i, b in enumerate(tr):
            x = b["x"].to(DEVICE); v = b["v"].to(DEVICE); y = b["labels"].to(DEVICE)
            with torch.autocast("cuda", dtype=AMP_DT, enabled=USE_AMP):
                logits = model(x, v)
                loss = crit(logits, y) / cfg["grad_accum"]
            scaler.scale(loss).backward()
            run_loss += loss.item() * cfg["grad_accum"]
            n_see += y.numel()
            step += 1
            if (i + 1) % cfg["grad_accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True)
            if step % max(1, cfg["log_every"]) == 0:
                wandb_log(wb, {"train/loss": run_loss / max(n_see, 1) * cfg["batch_size"], "step": step})
        sch.step()
        yv, pv, lv = predict_proba(model, va)
        vm = metrics_full(yv, pv)
        hist = {"epoch": ep + 1, "val": vm, "train_loss": run_loss / max(len(tr), 1)}
        history.append(hist)
        wandb_log(wb, {"epoch": ep + 1, "val/loss": lv, "val/acc": vm["acc"], "val/bal_acc": vm["balanced_acc"],
                       "val/precision": vm["precision"], "val/recall": vm["recall"], "val/f1": vm["f1"],
                       "val/auc": vm["roc_auc"], "val/pr_auc": vm["pr_auc"], "val/ece": vm["ece"],
                       "val/mcc": vm["mcc"], "lr": opt.param_groups[0]["lr"]})
        print(f"[{name}] ep{ep + 1:02d} loss={run_loss / max(len(tr), 1):.4f} "
              + " ".join(f"val/{k}={v:.4f}" for k, v in vm.items() if isinstance(v, float))[:240])
        if vm["acc"] > best_val:
            best_val = vm["acc"]
            bad = 0
            yt, pt, lt = predict_proba(model, te)
            best_te = metrics_full(yt, pt)
            os.makedirs(run_dir, exist_ok=True)
            ck = os.path.join(run_dir, f"{name}.pt")
            torch.save({"state_dict": model.state_dict(), "spec": spec,
                        "val": vm, "test": best_te, "epoch": ep + 1}, ck)
            np.save(os.path.join(run_dir, f"{name}_test_y.npy"), yt)
            np.save(os.path.join(run_dir, f"{name}_test_p.npy"), pt)
        else:
            bad += 1
            if bad >= cfg["patience"]:
                print(f"[{name}] early stop at ep{ep + 1}")
                break
    sec_ep = (time.time() - t0) / max(ep + 1, 1)
    res = {"name": name, "spec": spec, "params": npar, "best_ep": int(ep + 1), "sec_ep": sec_ep,
           "val": fmt_metric_row(name, {"acc": best_val}) | {"val_roc_auc": (best_te or {}).get("roc_auc")},
           "test": fmt_metric_row(name, best_te) if best_te else None,
           "status": "ok"}
    return res, history


In [18]:
def probe(model, loader, bs):
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    opt = torch.optim.AdamW(model.parameters(), 1e-4)
    npar = count_params(model)
    try:
        b = next(iter(loader))
        x = b["x"].to(DEVICE); v = b["v"].to(DEVICE); y = b["labels"].to(DEVICE)
        if USE_AMP:
            with torch.autocast("cuda", dtype=AMP_DT):
                loss = F.cross_entropy(model(x, v), y)
        else:
            loss = F.cross_entropy(model(x, v), y)
        loss.backward(); opt.step()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        vram = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
        del model, opt, x, v, y, loss
        torch.cuda.empty_cache()
        return {"params": npar, "vram": vram, "status": "ok", "bs": bs}
    except RuntimeError as e:
        torch.cuda.empty_cache()
        if bs > 1 and "out of memory" in str(e).lower():
            print("  [probe] OOM at bs", bs, "-> retry bs1")
            return probe(model, loader, 1)
        st = "OOM" if "out of memory" in str(e).lower() else "ERR"
        print("  [probe]", st, type(e).__name__, str(e)[:200])
        return {"params": npar, "vram": float("nan"), "status": st, "bs": bs}

def grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=enabled)

def summarize_run_row(name, spec, npar, best_ep, sec_ep, best_te):
    return {"name": name, "spec": spec, "params": npar, "best_ep": int(best_ep),
            "sec_ep": sec_ep, "test": fmt_metric_row(name, best_te) if best_te else None,
            "status": "ok"}

def train_one(cfg, spec, loaders, run_dir, wb=None):
    set_seed()
    name = spec["name"]
    tr, va, te = loaders
    model = build_model(cfg, spec).to(DEVICE)
    npar = count_params(model)
    eff = cfg["batch_size"] * cfg["grad_accum"]
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"] * (eff / 4) ** 0.5, weight_decay=cfg["wd"])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    scaler = grad_scaler(USE_AMP and not BF16)
    crit = nn.CrossEntropyLoss()
    best_val, best_te, bad, step = -1.0, None, 0, 0
    history = []
    t0 = time.time()
    print(f"[{name}] params={npar:.2f}M lr={opt.param_groups[0]['lr']:.2e}")
    for ep in range(cfg["epochs"]):
        model.train()
        opt.zero_grad(set_to_none=True)
        run_loss = 0.0
        for i, b in enumerate(tr):
            x = b["x"].to(DEVICE); v = b["v"].to(DEVICE); y = b["labels"].to(DEVICE)
            with torch.autocast("cuda", dtype=AMP_DT, enabled=USE_AMP):
                logits = model(x, v)
                loss = crit(logits, y) / cfg["grad_accum"]
            scaler.scale(loss).backward()
            run_loss += loss.item() * cfg["grad_accum"]
            step += 1
            if (i + 1) % cfg["grad_accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True)
            if step % max(1, cfg["log_every"]) == 0:
                wandb_log(wb, {"train/loss": run_loss / max(i + 1, 1), "step": step})
        sch.step()
        yv, pv, lv = predict_proba(model, va)
        vm = metrics_full(yv, pv)
        history.append({"epoch": ep + 1, "val_acc": vm["acc"], "val_f1": vm["f1"], "val_auc": vm["roc_auc"]})
        wandb_log(wb, {"epoch": ep + 1, "val/loss": lv, "val/acc": vm["acc"], "val/bal_acc": vm["balanced_acc"],
                       "val/precision": vm["precision"], "val/recall": vm["recall"], "val/specificity": vm["specificity"],
                       "val/f1": vm["f1"], "val/auc": vm["roc_auc"], "val/pr_auc": vm["pr_auc"], "val/ece": vm["ece"],
                       "val/mcc": vm["mcc"], "lr": opt.param_groups[0]["lr"]})
        line = " ".join(f"val/{k}={v:.4f}" for k, v in vm.items() if isinstance(v, float))
        print(f"[{name}] ep{ep + 1:02d} loss={run_loss / max(len(tr), 1):.4f} {line[:220]}")
        if vm["acc"] > best_val:
            best_val = vm["acc"]
            bad = 0
            yt, pt, lt = predict_proba(model, te)
            best_te = metrics_full(yt, pt)
            os.makedirs(run_dir, exist_ok=True)
            torch.save({"state_dict": model.state_dict(), "spec": spec, "epoch": ep + 1,
                        "test": best_te, "val": vm}, os.path.join(run_dir, f"{name}.pt"))
            np.save(os.path.join(run_dir, f"{name}_test_y.npy"), yt)
            np.save(os.path.join(run_dir, f"{name}_test_p.npy"), pt)
            np.save(os.path.join(run_dir, f"{name}_val_y.npy"), yv)
            np.save(os.path.join(run_dir, f"{name}_val_p.npy"), pv)
            wandb_log(wb, {"test/acc": best_te["acc"], "test/bal_acc": best_te["balanced_acc"],
                           "test/f1": best_te["f1"], "test/auc": best_te["roc_auc"],
                           "test/pr_auc": best_te["pr_auc"], "test/mcc": best_te["mcc"], "test/ece": best_te["ece"]})
        else:
            bad += 1
            if bad >= cfg["patience"]:
                print(f"[{name}] early stop at ep{ep + 1}")
                break
    sec_ep = (time.time() - t0) / max(ep + 1, 1)
    row = summarize_run_row(name, spec, npar, ep + 1, sec_ep, best_te)
    try:
        jdump({"history": history, "spec": spec, "val_acc_best": best_val},
              os.path.join(run_dir, f"{name}_history.json"))
    except Exception:
        pass
    return row, history


In [19]:
def run_summary_figures(name):
    ck_dir = os.path.join(SAVE_DIR, "ckpts")
    yp = os.path.join(ck_dir, f"{name}_test_y.npy")
    pp = os.path.join(ck_dir, f"{name}_test_p.npy")
    if not (os.path.isfile(yp) and os.path.isfile(pp)):
        return []
    yt = np.load(yp)
    pt = np.load(pp)
    out_dir = os.path.join(SAVE_DIR, "figures", name)
    os.makedirs(out_dir, exist_ok=True)
    paths = [plot_curves(yt, pt, os.path.join(out_dir, "test_roc_pr_calib.png")),
             plot_cm(yt, (pt >= 0.5).astype(int), os.path.join(out_dir, "confusion_test.png"))]
    hist = jload(os.path.join(ck_dir, f"{name}_history.json"), default={}).get("history", [])
    if hist:
        plot_history(hist, os.path.join(out_dir, "history.png"))
        paths.append(os.path.join(out_dir, "history.png"))
    for p in paths:
        copy_to_drive(p, sub=os.path.join("figures", name))
    print(f"[figures] {name}: " + ", ".join(os.path.basename(p) for p in paths) + " -> Drive figures/{name}")
    return paths


## 6. Sweep driver (resume-safe, per-tier)

Every spec carries `res3d` / `res2d` / `batch_size` / `epochs` overrides. Resolution policy:
**keep the raw 200^3 whenever the architecture allows it** (all from-scratch + MONAI conv encoders
run at `res3d=200` = no resampling at all); only transformer/ViT rows need divisibility so they run
at 160-192. Loaders are lazy and cached per `(res3d, res2d, bs, load3d, loadviews)`; for single-branch
rows the unused modality is **not even read** (big I/O saving). Each finished run is appended to
`results.json` (local + Drive) -> re-running resumes instead of restarting, and per-run figure PNGs
(test ROC/PR/calibration, confusion matrix, val history) are written under `figures/<run>` + Drive +
wandb the moment the run finishes.


In [20]:
RESULTS_LOCAL = os.path.join(SAVE_DIR, "results.json")
RESULTS_DRIVE = os.path.join(DRIVE_DIR, "results.json")

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

def load_rows():
    for p in (RESULTS_LOCAL, RESULTS_DRIVE):
        r = jload(p)
        if r:
            return r
    return []

def save_rows(rows):
    jdump(rows, RESULTS_LOCAL)
    copy_to_drive(RESULTS_LOCAL)
    print(f"[save] {len(rows)} rows -> {RESULTS_DRIVE}")

_LOADER_CACHE = {}

def get_loaders(cfg, spec):
    res3d = int(spec.get("res3d", MODEL_RES3D_DEFAULT))
    res2d = int(spec.get("res2d", RES2D))
    bs = int(spec.get("batch_size", cfg["batch_size"]))
    load3d = spec.get("fusion") != "single2d"
    loadviews = spec.get("fusion") not in ("single3d",)
    key = (res3d, res2d, bs, load3d, loadviews)
    if key in _LOADER_CACHE:
        return _LOADER_CACHE[key]
    cfg2 = dict(cfg)
    cfg2["batch_size"] = bs
    loaders = build_loaders(res3d=res3d, res2d=res2d, bs=bs, max_train=cfg.get("max_train", 0),
                            load3d=load3d, loadviews=loadviews)
    _LOADER_CACHE[key] = loaders
    return loaders

def print_row_short(r):
    te = r.get("test") or {}
    return (f"{r['name']:28s} par={r.get('params', float('nan')):7.2f}M "
            f"ep={r.get('best_ep', -1):3d} "
            f"acc={te.get('acc', float('nan')):.4f} f1={te.get('f1', float('nan')):.4f} "
            f"auc={te.get('roc_auc', float('nan')):.4f} spec={te.get('specificity', float('nan')):.4f} "
            f"mcc={te.get('mcc', float('nan')):.4f} ece={te.get('ece', float('nan')):.4f}")

def sweep_tier(cfg, tier, tier_label, wb_master=None):
    rows = load_rows()
    done = {r["name"] for r in rows if r.get("status") == "ok" and r.get("test")}
    ran = []
    for spec in tier:
        if not spec.get("enabled", True):
            print(f"[skip-off] {spec['name']} (disabled)")
            continue
        name = spec["name"]
        if name in done:
            print(f"[resume] {name} already done")
            continue
        cfg2 = dict(cfg)
        cfg2["epochs"] = int(spec.get("epochs", cfg["epochs"]))
        cfg2["batch_size"] = int(spec.get("batch_size", cfg["batch_size"]))
        tr, va, te = get_loaders(cfg, spec)
        model = build_model(cfg2, spec).to(DEVICE)
        pr = probe(model, tr, cfg2["batch_size"])
        del model
        torch.cuda.empty_cache()
        if pr["status"] != "ok":
            row = {"name": name, "spec": to_jsonable(spec), "params": pr["params"], "vram": pr["vram"],
                   "status": pr["status"], "bs": pr["bs"]}
            rows.append(row); save_rows(rows); ran.append(row)
            print(f"[probe] {name}: {pr['status']}")
            continue
        wb = init_wandb(name, config=to_jsonable({"name": name, "fusion": spec["fusion"],
                                                  "enc3d": spec.get("enc3d", {}).get("kind"),
                                                  "enc2d": spec.get("enc2d", {}).get("name") or spec.get("enc2d", {}).get("kind"),
                                                  "res3d": spec.get("res3d"), "res2d": spec.get("res2d"),
                                                  "epochs": cfg2["epochs"], "bs": cfg2["batch_size"]}))
        row = None
        figs = []
        try:
            print(f"[run] {name} params={pr['params']:.2f}M vram~{pr['vram']:.1f}GB bs={pr['bs']}")
            row, _hist = train_one(cfg2, spec, (tr, va, te), os.path.join(SAVE_DIR, "ckpts"), wb=wb)
            if row and row.get("status") == "ok" and row.get("test"):
                figs = run_summary_figures(name)
                if wb is not None and figs:
                    try:
                        wb.log({"run/final": [__import__("wandb").Image(p) for p in figs]})
                    except Exception as e:
                        print("[wandb] figure log fail", e)
        except Exception as e:
            print(f"[run] {name} FAILED: {type(e).__name__}: {str(e)[:300]}")
            row = {"name": name, "spec": to_jsonable(spec), "params": pr["params"],
                   "status": "ERR", "err": str(e)[:300]}
        finally:
            if wb is not None:
                try:
                    if row and row.get("test"):
                        wb.summary.update(row["test"])
                    wb.finish()
                except Exception:
                    pass
        rows.append(row)
        save_rows(rows)
        if row and row.get("status") == "ok" and row.get("test"):
            print("  ", print_row_short(row))
        elif row:
            print("  ", row.get("status"), name)
    return load_rows()


Sweep definitions. `enabled` selects the default rows (curated so one A100 session finishes); set
more rows to `enabled=True` (or toggle `RUN_TIER_x`) to widen the sweep. Honest guidance:
- Tier A (3D single branch): conv rows run at the **raw 200^3** (max resolution) ~ 10-25 min/run;
  transformer rows at 160^3 ~ 10-30 min/run.
- Tier B (2D pretrained single view @224): fast, ~2-6 min/run after ImageNet weights download.
- Tier C (fusion): conv-3D @ raw 200 + 3x 2D backbones @224.
When a run finishes you immediately get figures at `Drive/.../multiview_sota_sweep/figures/<run>/`
(test ROC/PR/calibration + confusion + val history) plus wandb images/metrics. Add `"epochs": 6` to
heavy rows to halve cost.


In [21]:
RUN_TIER_A = os.environ.get("MVX_TIER_A", "1") == "1"
RUN_TIER_B = os.environ.get("MVX_TIER_B", "1") == "1"
RUN_TIER_C = os.environ.get("MVX_TIER_C", "1") == "1"

# Resolution policy: conv-type 3D backbones run at the RAW 200^3 (no resize at all). Only the
# transformer/ViT rows drop to 160/192 because patch/window sizes require divisibility (and VRAM);
# keep them disabled unless you want the transformer comparison.
TIER_A_3D_SINGLE = [
    {"name": "single3d-cnn3d", "fusion": "single3d", "enc3d": {"kind": "cnn3d"},
     "res3d": 96, "enabled": True, "note": "reference micro-resnet3d (baseline) @ 96^3"},
    {"name": "single3d-convnext3d", "fusion": "single3d", "enc3d": {"kind": "convnext3d"},
     "res3d": 96, "enabled": True, "note": "ConvNeXt-style 3D @ 96^3"},
    {"name": "single3d-resxt3d", "fusion": "single3d", "enc3d": {"kind": "resxt3d"},
     "res3d": 96, "enabled": True, "note": "ResNeXt-style 3D @ 96^3"},
    {"name": "single3d-vit3d", "fusion": "single3d", "enc3d": {"kind": "vit3d"},
     "res3d": 96, "enabled": True, "note": "patch-16 ViT-3D @ 96^3 (96 % 16 == 0)"},
    {"name": "single3d-segresnet", "fusion": "single3d", "enc3d": {"kind": "segresnet", "params": {"init_filters": 16}},
     "res3d": 96, "enabled": True, "note": "MONAI SegResNet encoder @ 96^3"},
    {"name": "single3d-dynunet16", "fusion": "single3d", "enc3d": {"kind": "dynunet", "params": {"filters": (16, 32, 64, 128, 256)}},
     "res3d": 96, "enabled": True, "note": "MONAI DynUNet encoder @ 96^3"},
    {"name": "single3d-vnet", "fusion": "single3d", "enc3d": {"kind": "vnet"},
     "res3d": 96, "enabled": True, "note": "V-Net encoder @ 96^3"},
    {"name": "single3d-unetr", "fusion": "single3d", "enc3d": {"kind": "unetr", "params": {"hidden_size": 192, "img_size": (96, 96, 96)}},
     "res3d": 96, "enabled": True, "note": "UNETR ViT encoder @ 96^3 (img_size must equal res)"},
    {"name": "single3d-swinunetr", "fusion": "single3d", "enc3d": {"kind": "swinunetr", "params": {"feature_size": 24}},
     "res3d": 96, "enabled": True, "epochs": 10, "note": "SwinUNETR @ 96^3 (96 % 32 == 0)"},
    {"name": "single3d-mednet10", "fusion": "single3d", "enc3d": {"kind": "mednet10", "params": {"pretrained": True}},
     "res3d": 96, "enabled": True, "epochs": 10, "note": "MedicalNet 3D ResNet10 @ 96^3"},
    {"name": "single3d-mednet50", "fusion": "single3d", "enc3d": {"kind": "mednet50", "params": {"pretrained": True}},
     "res3d": 96, "batch_size": 1, "enabled": True, "epochs": 6, "note": "ResNet50-3D heavy @ 96^3"},
    {"name": "single3d-3dino", "fusion": "single3d", "enc3d": {"kind": "3dino"},
     "res3d": 96, "batch_size": 1, "enabled": True, "epochs": 10,
     "note": "3DINO-ViT-L frozen (gated: accept terms + HF_TOKEN) @ 96^3; pos-embed error -> set res3d 112"},
]

TIER_B_2D_SINGLE = [
    {"name": f"single2d-tiny2d-{i}-{VIEWS[i]}", "fusion": "single2d", "view": i,
     "enc2d": {"kind": "tiny2d"}, "res3d": 64, "res2d": 224, "enabled": True}
    for i in range(3)
] + [
    {"name": f"single2d-{name}-slabaip", "fusion": "single2d", "view": 1,
     "enc2d": {"kind": "timm", "name": name, "pretrained": True}, "res3d": 64, "res2d": 224,
     "enabled": name in ("convnextv2_tiny", "deit3_small_patch16_224", "maxvit_tiny_rw_224")}
    for name in ["convnextv2_tiny", "convnext_tiny", "swin_tiny_patch4_window7_224",
                 "vit_base_patch16_224", "deit3_small_patch16_224", "maxvit_tiny_rw_224"]
]

_FUSION_OPS = ["concat", "add", "mul", "attn", "crossgate", "mamba", "film"]

TIER_C_FUSION = [
    {"name": f"fusion-{op}-convnext3d-convnextv2tiny", "fusion": op,
     "enc3d": {"kind": "convnext3d"},
     "enc2d": {"kind": "timm", "name": "convnextv2_tiny", "pretrained": True},
     "res3d": 200, "res2d": 224, "epochs": 10,
     "enabled": op in ("concat", "attn", "crossgate", "mamba", "film", "add")}
    for op in _FUSION_OPS
]

if SMOKE:
    RUN_TIER_A = RUN_TIER_B = RUN_TIER_C = True
    for lst in (TIER_A_3D_SINGLE, TIER_B_2D_SINGLE, TIER_C_FUSION):
        for sp in lst:
            sp["enabled"] = sp.get("name") in (
                "single3d-cnn3d", "single3d-convnext3d", "single3d-mednet10",
                "single2d-tiny2d-1-slab_aip", "fusion-attn-convnext3d-convnextv2tiny",
                "fusion-mamba-convnext3d-convnextv2tiny")


## 7. X-AI explainability

Applied to a trained model + a handful of held-out test samples (balanced per class). Produces, per
sample, saved PNGs + a machine/human-readable explanation block:
- **Grad-CAM 3D** on the last deep conv stage (or patch-embed conv for transformer encoders) overlaid on
  depth B-scans,
- **Grad-CAM 2D** per en-face view,
- **Occlusion sensitivity** (3D, coarse grid, optional, conv encoders only),
- **Integrated gradients** on a 2D view (and optionally the 3D input downsampled to 64^3),
- **Branch importance**: fusion attention weights (attn/crossgate ops) or leave-one-branch-out drop for
  the other ops — answers *which stream drives the decision*,
- **LIME on 2D view** (superpixel perturbations around the best single-2D model, optional).

All artefacts are copied to Drive (`.../Thesis/multiview_sota_sweep/xai_<run>`), PNGs logged to wandb,
and a Markdown explanation is written + printed.


In [22]:
XAI_DIR = os.path.join(SAVE_DIR, "xai")
os.makedirs(XAI_DIR, exist_ok=True)

def _numpy(x):
    return x.detach().cpu().numpy()

def _heat_to(x, size):
    t = torch.from_numpy(np.asarray(x, dtype=np.float32))[None, None]
    t = F.interpolate(t, size=size, mode="trilinear" if len(size) == 3 else "bilinear", align_corners=False)
    h = _numpy(t[0, 0])
    h = h - h.min()
    if h.max() > 0:
        h = h / h.max()
    return h

def gradcam_map(model, module, x, v, target, eps=1e-6):
    model.eval()
    store = {}
    def fa(m, i, o):
        store["act"] = o
    def fb(m, gi, go):
        store["grad"] = go[0]
    h1 = module.register_forward_hook(fa)
    h2 = module.register_full_backward_hook(fb)
    xx = x.clone().float().requires_grad_(True)
    vv = v.clone().float().requires_grad_(True)
    logits = model(xx, vv)
    score = logits[0, target]
    model.zero_grad(set_to_none=True)
    score.backward(retain_graph=True)
    act = store["act"]
    grad = store["grad"]
    if isinstance(act, (tuple, list)):
        act = act[0]
    if isinstance(grad, (tuple, list)):
        grad = grad[0]
    w = grad.mean(dim=tuple(range(2, grad.ndim)), keepdim=True)
    cam = F.relu((w * act).sum(dim=1, keepdim=True))
    h1.remove(); h2.remove()
    model.zero_grad(set_to_none=True)
    return _numpy(cam[0, 0]), _numpy(logits[0])

def occl_importance(model, x, v, target, grid=4, patch=None, base=None):
    model.eval()
    dims = x.shape[2:]
    grid = max(2, int(grid))
    patch = tuple(max(1, int(d // grid)) for d in dims) if patch is None else patch
    xx = x.clone().float()
    vv = v.clone().float()
    with torch.no_grad():
        base = float(torch.softmax(model(xx, vv).float(), 1)[0, target]) if base is None else base
    drop = np.zeros((grid,) * len(dims))
    for idx in np.ndindex(*([grid] * len(dims))):
        masked = xx.clone()
        sl = tuple(slice(i * p, min((i + 1) * p, d)) for i, p, d in zip(idx, patch, dims))
        masked[0, 0][sl] = 0.0
        with torch.no_grad():
            p = float(torch.softmax(model(masked, vv).float(), 1)[0, target])
        drop[idx] = max(0.0, base - p)
    return drop, base

def integrated_grad(model, x_in, target, steps=12, channels_first_spatial=None):
    model.eval()
    xb = x_in.clone().float()
    if channels_first_spatial is not None:
        xb = xb.unsqueeze(0) if xb.ndim == 2 else xb
        xb = F.interpolate(xb, size=channels_first_spatial, mode="trilinear", align_corners=False)
        xb = xb.squeeze(0)
    xb = xb.unsqueeze(0)
    vv = torch.zeros(1, len(VIEWS), RES2D, RES2D, device=x_in.device)
    bg = torch.zeros_like(xb)
    acc = torch.zeros_like(xb)
    for k in range(1, steps + 1):
        xt = bg + (xb - bg) * (k / steps)
        xt = xt.clone().requires_grad_(True)
        logits = model(xt, vv)
        logits[0, target].backward()
        acc += xt.grad.detach()
        model.zero_grad(set_to_none=True)
    attr = (xb - bg) * acc / steps
    return _numpy(attr[0, 0])

def ig_2d_view(model, x, v, idx_view, target, steps=8):
    model.eval()
    xb = x.clone().float().detach()
    vb = v.clone().float()
    acc = torch.zeros_like(vb)
    for k in range(1, steps + 1):
        vv = (vb * (k / steps)).clone().requires_grad_(True)
        logits = model(xb, vv)
        logits[0, target].backward()
        acc += vv.grad.detach()
        model.zero_grad(set_to_none=True)
    return _numpy((vb * acc / steps)[0, idx_view])


In [23]:
import skimage

def lime_2d(model, v, idx_view, target, n_perturb=160, n_feat=12, seed=SEED):
    from skimage.segmentation import slic, mark_boundaries
    from sklearn import linear_model
    rng = np.random.default_rng(seed)
    img = _numpy(v[idx_view])
    seg = slic(img, n_segments=48, compactness=0.15, channel_axis=None)
    n = int(seg.max()) + 1
    xb = torch.zeros(1, 1, img.shape[0], img.shape[1], device=v.device)
    pert = rng.random((n_perturb, n)) > 0.5
    pert[0] = False
    feats, probs = [], []
    vb = v.clone().float()
    for row in pert:
        mimg = img.copy()
        for j in np.where(row)[0]:
            mimg[seg == j] = float(img.mean())
        vv = vb.clone()
        vv[idx_view] = torch.from_numpy(mimg).to(v.device)
        with torch.no_grad():
            p = float(torch.softmax(model(xb, vv.unsqueeze(0)).float(), 1)[0, target])
        feats.append(row.astype(np.float64))
        probs.append(p)
    feats = np.stack(feats)
    probs = np.array(probs)
    w = np.linalg.lstsq(feats, probs, rcond=None)[0]
    top = np.argsort(-np.abs(w))[:n_feat]
    imp = np.zeros_like(img, dtype=np.float32)
    for j in top:
        imp[seg == j] = float(w[j])
    return imp, mark_boundaries(img, seg, mode="outer")


In [24]:
def fusion_logits(model, toks):
    op = model.mode
    if op == "concat":
        return model.head(toks.reshape(toks.shape[0], -1))
    if op == "add":
        return model.head(toks.sum(dim=1))
    if op == "mul":
        return model.head(torch.prod(toks + 1e-6, dim=1))
    if op == "attn":
        return model.head(model.fusion(toks))
    if op == "crossgate":
        return model.head(model.fusion(toks[:, 0], toks[:, 1:]))
    if op == "mamba":
        return model.fusion(toks)
    if op == "film":
        return model.fusion(toks)
    return model.head(toks.reshape(toks.shape[0], -1))

@torch.no_grad()
def branch_importance(model, x, v, target):
    model.eval()
    names = list(model.names)
    es = model.embed(x, v)
    toks = torch.stack([p(e) for p, e in zip(model.projs, es)], dim=1)
    base = float(torch.softmax(fusion_logits(model, toks).float(), 1)[0, target])
    drops = []
    for j in range(len(names)):
        tm = toks.clone()
        tm[:, j] = 0.0
        p = float(torch.softmax(fusion_logits(model, tm).float(), 1)[0, target])
        drops.append(max(0.0, base - p))
    return names, drops, base

def fusion_attention_weights(model):
    if model.mode == "attn" and getattr(model.fusion, "last_cls_w", None) is not None:
        return list(model.names), _numpy(model.fusion.last_cls_w)
    if model.mode == "crossgate" and getattr(model.fusion, "last_w", None) is not None:
        w = _numpy(model.fusion.last_w).mean(0)
        return list(model.names)[1:], w
    return None

def pick_test_samples(test_loader, per_class=2, seed=SEED):
    rng = np.random.default_rng(seed)
    ds = test_loader.dataset
    idx = np.arange(len(ds))
    labs = [int(ds.labels[i]) for i in range(len(ds))] if hasattr(ds, "labels") else None
    chosen = []
    if labs is None:
        return []
    for c in sorted(set(labs)):
        cand = idx[np.asarray(labs) == c]
        cand = rng.choice(cand, size=min(per_class, len(cand)), replace=False)
        chosen.extend(int(i) for i in cand)
    return chosen

def sample_tensor(ds, i):
    d = ds[i]
    x = d["x"].unsqueeze(0).to(DEVICE).float()
    v = d["v"].unsqueeze(0).to(DEVICE).float()
    y = int(d["labels"])
    return x, v, y

def render_vol_slices(img3d, title, path):
    D = img3d.shape[2]
    ks = np.linspace(int(D * 0.2), int(D * 0.8), 6).astype(int)
    ks = sorted(set(int(min(max(k, 0), D - 1)) for k in ks))
    fig, axes = plt.subplots(1, len(ks), figsize=(2.6 * len(ks), 2.6))
    for ax, k in zip(axes if len(ks) > 1 else [axes], ks):
        sl = np.asarray(img3d[..., k]).astype(np.float32) if img3d.ndim == 3 else np.asarray(img3d[k])
        sl = sl - sl.min()
        if sl.max() > 0:
            sl = sl / sl.max()
        ax.imshow(sl, cmap="gray")
        ax.set_title(f"depth {k}")
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)

def render_heat_slices(xv, heat, title, path):
    D = xv.shape[2]
    ks = np.linspace(int(D * 0.2), int(D * 0.8), 6).astype(int)
    ks = sorted(set(int(min(max(k, 0), D - 1)) for k in ks))
    fig, axes = plt.subplots(2, len(ks), figsize=(2.6 * len(ks), 5.2))
    for j, k in enumerate(ks):
        sl = np.asarray(xv[..., k]).astype(np.float32)
        p1, p99 = np.percentile(sl, 1), np.percentile(sl, 99)
        axes[0, j].imshow(sl, cmap="gray", vmin=p1, vmax=p99)
        axes[0, j].axis("off")
        h = np.asarray(heat[..., k]).astype(np.float32)
        axes[1, j].imshow(sl, cmap="gray", vmin=p1, vmax=p99)
        axes[1, j].imshow(h, cmap="jet", alpha=0.45)
        axes[1, j].axis("off")
        axes[1, j].set_title(f"depth {k}")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


In [25]:
def render_view_heatmap(view_img, heat, view_name, title, path):
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 4.2))
    axes[0].imshow(view_img, cmap="gray"); axes[0].set_title("input"); axes[0].axis("off")
    axes[1].imshow(view_img, cmap="gray")
    axes[1].imshow(heat, cmap="jet", alpha=0.45); axes[1].set_title(view_name); axes[1].axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)

def explain_sample(model, cfg, x, v, y, sample_tag, out_dir, opts):
    target = int(y)
    lines = []
    with torch.no_grad():
        p = float(torch.softmax(model(x, v).float(), 1)[0, 1])
    pred = 1 if p >= 0.5 else 0
    tag = f"true={y} pred={pred} p_glc={p:.3f}"
    if opts.get("gradcam3d", True) and model.enc3d is not None:
        mod = model.gcam3d_module()
        if mod is not None:
            cam, logits = gradcam_map(model, mod, x, v, target)
            heat = _heat_to(cam, tuple(x.shape[2:]))
            render_heat_slices(_numpy(x[0, 0]), heat, f"Grad-CAM 3D | {sample_tag} | {tag}", os.path.join(out_dir, f"{sample_tag}_gradcam3d.png"))
            peak = float(heat.max())
            lines.append(f"Grad-CAM 3D max={peak:.2f} (peak located near depth {int(np.unravel_index(heat.argmax(), heat.shape)[2])}).")
    if opts.get("gradcam2d", True) and len(model.enc2ds):
        view_range = [model.view_idx] if model.mode == "single2d" else list(range(len(model.enc2ds)))
        for i in view_range:
            mod = model.gcam2d_module(i)
            if mod is None:
                continue
            cam, _ = gradcam_map(model, mod, x, v, target)
            heat = _heat_to(cam, (v.shape[-2], v.shape[-1]))
            vi = _numpy(v[0, i])
            p1, p99 = np.percentile(vi, 1), np.percentile(vi, 99)
            render_view_heatmap(np.clip((vi - p1) / max(p99 - p1, 1e-6), 0, 1), heat, VIEWS[i],
                                f"Grad-CAM 2D {VIEWS[i]} | {sample_tag} | {tag}",
                                os.path.join(out_dir, f"{sample_tag}_gradcam2d_{VIEWS[i]}.png"))
        lines.append("Grad-CAM 2D: heatmaps written for the relevant en-face views.")
    if opts.get("occlusion", True) and model.enc3d is not None:
        try:
            grid = int(opts.get("occlusion_grid", 3))
            drop, base = occl_importance(model, x, v, target, grid=grid)
            heat = _heat_to(drop, tuple(x.shape[2:]))
            render_heat_slices(_numpy(x[0, 0]), heat, f"Occlusion | {sample_tag} | {tag}",
                               os.path.join(out_dir, f"{sample_tag}_occlusion3d.png"))
            lines.append(f"Occlusion3D: base p={base:.3f}, strongest drop region at "
                         f"{tuple(int(j) for j in np.unravel_index(drop.argmax(), drop.shape))} (grid cell).")
        except Exception as e:
            lines.append(f"occlusion3d skipped ({type(e).__name__})")
    return "\n".join(lines)


In [26]:
def plot_history(hist, path):
    if not hist:
        return None
    epochs = [h.get("epoch", i + 1) for i, h in enumerate(hist)]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, [h.get("val_acc", float("nan")) for h in hist], "o-", label="val acc")
    ax.plot(epochs, [h.get("val_f1", float("nan")) for h in hist], "s--", label="val f1")
    ax.plot(epochs, [h.get("val_auc", float("nan")) for h in hist], "d-.", label="val auc")
    ax.set_xlabel("epoch"); ax.set_ylabel("metric")
    ax.set_title("validation history"); ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)
    return path

def load_ckpt_row(name):
    p = os.path.join(SAVE_DIR, "ckpts", f"{name}.pt")
    return torch.load(p, map_location=DEVICE) if os.path.isfile(p) else None

def rebuild_model_from_row(cfg, row):
    spec = row["spec"]
    model = build_model(cfg, spec).to(DEVICE)
    ck = load_ckpt_row(row["name"])
    if ck:
        model.load_state_dict(ck["state_dict"])
    model.eval()
    return model, spec

def choose_winner(rows):
    ok = [r for r in rows if r.get("status") == "ok" and r.get("test")]
    if not ok:
        return None
    ok = sorted(ok, key=lambda r: -(r["test"].get("roc_auc") or 0.0))
    return ok[0]

def best_single2d(rows):
    cand = [r for r in rows if r.get("status") == "ok" and r.get("spec", {}).get("fusion") == "single2d"]
    cand = sorted(cand, key=lambda r: -(r["test"].get("roc_auc") or 0.0))
    return cand[0] if cand else None


In [27]:
def write_xai_markdown(path, name, samples_txt, branch_lines, metrics_lines, figlist):
    body = [f"# X-AI report: {name}", ""]
    body += ["## Per-sample explanation", ""] + samples_txt + ["", "## Branch importance", ""]
    body += branch_lines + ["", "## Test metrics (best epoch)", ""] + metrics_lines
    body += ["", "## Figures", ""] + [f"- {f}" for f in figlist]
    txt = "\n".join(body)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(txt)
    return txt

def run_xai_pipeline(cfg, row, opts=None):
    name = row["name"]
    opts = opts or {"gradcam3d": True, "gradcam2d": True, "occlusion": True,
                    "occlusion_grid": 3, "ig2d": True, "branch_ablation": True, "lime": True}
    out_dir = os.path.join(XAI_DIR, name)
    os.makedirs(out_dir, exist_ok=True)
    model, spec = rebuild_model_from_row(cfg, row)
    tr, va, te = get_loaders(cfg, spec)
    yt, pt, _ = predict_proba(model, te)
    m = metrics_full(yt, pt)
    cm_png = os.path.join(out_dir, "confusion_test.png")
    plot_cm(yt, (pt >= 0.5).astype(int), cm_png)
    curves = os.path.join(out_dir, "roc_pr_calib.png")
    plot_curves(yt, pt, curves)
    hist = jload(os.path.join(SAVE_DIR, "ckpts", f"{name}_history.json"), default={})
    hist_png = os.path.join(XAI_DIR, f"{name}_history.png")
    plot_history(hist.get("history", []), hist_png)
    figlist = [os.path.basename(p) for p in (cm_png, curves, hist_png)]
    boot_auc = bootstrap_ci(yt, pt, "roc_auc")
    boot_acc = bootstrap_ci(yt, pt, "acc")
    mlines = [f"- acc {m['acc']:.4f} (95% CI {boot_acc[0]:.3f}-{boot_acc[1]:.3f})",
              f"- balanced_acc {m['balanced_acc']:.4f}", f"- precision/PPV {m['precision']:.4f}",
              f"- recall/sens {m['recall']:.4f}", f"- specificity {m['specificity']:.4f}",
              f"- NPV {m['npv']:.4f}", f"- F1 {m['f1']:.4f} / F1-macro {m['f1_macro']:.4f}",
              f"- MCC {m['mcc']:.4f} | Kappa {m['kappa']:.4f} | Youden {m['youden']:.4f}",
              f"- ROC-AUC {m['roc_auc']:.4f} (95% CI {boot_auc[0]:.3f}-{boot_auc[1]:.3f})",
              f"- PR-AUC {m['pr_auc']:.4f} | ECE {m['ece']:.4f}",
              f"- Log-loss {m['logloss']:.4f} | Brier {m['brier']:.4f}",
              f"- confusion {m['tn']}/{m['fp']}/{m['fn']}/{m['tp']} (TN/FP/FN/TP)"]
    yv_arr = os.path.join(SAVE_DIR, "ckpts", f"{name}_val_y.npy")
    pv_arr = os.path.join(SAVE_DIR, "ckpts", f"{name}_val_p.npy")
    if os.path.isfile(yv_arr) and os.path.isfile(pv_arr):
        th, _j = optimal_youden_threshold(np.load(yv_arr), np.load(pv_arr))
        mt = metrics_at_threshold(yt, pt, th)
        mlines.append(f"- operating point: Youden threshold from val p>={th:.3f} -> test acc "
                      f"{mt['acc']:.4f} sens {mt['recall']:.4f} spec {mt['specificity']:.4f} "
                      f"PPV {mt['precision']:.4f} NPV {mt['npv']:.4f} F1 {mt['f1']:.4f}")
    samples_txt, sample_figs = [], []
    ds = te.dataset
    for si, i in enumerate(pick_test_samples(te, cfg.get("xai_samples_per_class", 1))):
        x, v, y = sample_tensor(ds, i)
        tag = f"sample{si}_idx{i}"
        txt = explain_sample(model, cfg, x, v, y, tag, out_dir, opts)
        samples_txt.append(f"### sample idx={i} true={y} -- {txt.replace(chr(10), ' ')}")
        sample_figs += [f for f in os.listdir(out_dir) if f.startswith(tag)]
    blines = []
    if opts.get("branch_ablation", True) and model.mode not in ("single3d", "single2d"):
        x, v, y = sample_tensor(ds, pick_test_samples(te, 1)[0])
        names, drops, base = branch_importance(model, x, v, int(y))
        for nm, d in zip(names, drops):
            blines.append(f"- branch {nm}: drop in P(class) when removed = {d:.4f} (base p {base:.3f})")
        fig, ax = plt.subplots(figsize=(6.5, 3.2))
        ax.barh(names[::-1], drops[::-1]); ax.set_xlabel("P-drop when branch zeroed")
        ax.set_title(f"Branch importance ({name})")
        fig.tight_layout()
        bip = os.path.join(out_dir, "branch_importance.png")
        fig.savefig(bip, dpi=130); plt.close(fig)
        figlist.append(os.path.basename(bip))
    elif opts.get("branch_ablation", True) and model.mode in ("single3d", "single2d"):
        blines = ["single-branch model - no fusion to ablate."]
    aw = fusion_attention_weights(model)
    if aw is not None:
        names, w = aw
        fig, ax = plt.subplots(figsize=(6.5, 3.2))
        ax.barh(names[::-1], w[::-1]); ax.set_xlabel("mean attention weight")
        ax.set_title(f"Fusion attention ({model.mode})")
        fig.tight_layout()
        awp = os.path.join(out_dir, "fusion_attention.png")
        fig.savefig(awp, dpi=130); plt.close(fig)
        figlist.append(os.path.basename(awp))
        blines.append("attention weights: " + ", ".join(f"{n}={float(w_):.3f}" for n, w_ in zip(names, w)))
    if opts.get("ig2d", True) and len(model.enc2ds):
        x, v, y = sample_tensor(ds, pick_test_samples(te, 1)[0])
        i = model.view_idx if model.mode == "single2d" else (1 if len(VIEWS) > 1 else 0)
        attr = ig_2d_view(model, x, v, i, int(y), steps=8)
        fig, ax = plt.subplots(figsize=(7, 3.4))
        ax.imshow(np.abs(attr), cmap="hot")
        ax.set_title(f"Integrated gradients | view {VIEWS[i]}")
        fig.colorbar(plt.gca().images[0], ax=ax)
        fig.tight_layout()
        igp = os.path.join(out_dir, "integrated_gradients_view.png")
        fig.savefig(igp, dpi=130); plt.close(fig)
        figlist.append(os.path.basename(igp))
        blines.append(f"IG2D computed on view {VIEWS[i]} (integrated gradients magnitude).")
    md_txt = write_xai_markdown(os.path.join(out_dir, "XAI_REPORT.md"), name, samples_txt, blines, mlines, figlist)
    print(md_txt)
    shutil.copy(os.path.join(XAI_DIR, f"{name}_history.png"), out_dir)
    return out_dir, m


## 8. Run the sweep (GPU). Resume-safe.

Rows are persisted after each run (local + Drive). To stop early: interrupt the cell; finished rows
survive. To widen: set extra rows `enabled=True` above (or flip `MVX_TIER_*` env vars).


In [ ]:
WINNER = None
rows = load_rows()

if not SMOKE and RUN_SWEEP:
    if RUN_TIER_A:
        print("\n===== TIER A: 3D single-branch backbones (all @ 96^3) =====")
        rows = sweep_tier(CFG, TIER_A_3D_SINGLE, "A")
    # if RUN_TIER_B:
    #     print("\n===== TIER B: 2D single-view (pretrained timm / from-scratch) =====")
    #     rows = sweep_tier(CFG, TIER_B_2D_SINGLE, "B")
    # if RUN_TIER_C:
    #     print("\n===== TIER C: multiview fusion (3D + 3x2D) fusion-op ablation =====")
    #     rows = sweep_tier(CFG, TIER_C_FUSION, "C")
    rows = load_rows()

    hdr = f"{'model':38s} {'pars':>8} {'ep':>4} {'acc':>7} {'bal':>7} {'f1':>7} {'auc':>7} {'spec':>7} {'mcc':>7} {'ece':>7}"
    print("\n=================== FULL RESULTS (test set, best epoch) ===================")
    print(hdr)
    for r in sorted(rows, key=lambda r: (r.get("status") == "ok", -(r.get("test") or {}).get("roc_auc") or 0)):
        te = r.get("test") or {}
        st = r.get("status", "?")
        print(f"{r['name']:38s} {r.get('params', float('nan')):>8.2f} {r.get('best_ep', -1):>4d} "
              f"{te.get('acc', float('nan')):>7.4f} {te.get('balanced_acc', float('nan')):>7.4f} "
              f"{te.get('f1', float('nan')):>7.4f} {te.get('roc_auc', float('nan')):>7.4f} "
              f"{te.get('specificity', float('nan')):>7.4f} {te.get('mcc', float('nan')):>7.4f} "
              f"{te.get('ece', float('nan')):>7.4f}  {st}")
    WINNER = choose_winner(rows)
    if WINNER:
        print("\n>> WINNER (max test ROC-AUC):")
        print_row_short(WINNER)
    else:
        print("\n>> no completed run with test metrics yet")



===== TIER B: 2D single-view (pretrained timm / from-scratch) =====
[data] consolidated arrays already present in /content/glaucoma_hf_200
[ds] Training n=2100 res3d=64 res2d=224 load3d=False loadviews=True
[ds] Validation n=300 res3d=64 res2d=224 load3d=False loadviews=True
[ds] Test n=900 res3d=64 res2d=224 load3d=False loadviews=True
[run] single2d-tiny2d-0-aip_full params=2.79M vram~0.6GB bs=2
[single2d-tiny2d-0-aip_full] params=2.79M lr=4.00e-04
[single2d-tiny2d-0-aip_full] ep01 loss=0.7458 val/acc=0.4133 val/balanced_acc=0.5000 val/precision=0.0000 val/recall=0.0000 val/specificity=1.0000 val/npv=0.4133 val/f1=0.0000 val/f1_macro=0.2925 val/mcc=0.0000 val/roc_auc=0.5993 val/pr_auc=0.6768 val/ece=0.2017 val
[single2d-tiny2d-0-aip_full] ep02 loss=0.7104 val/acc=0.5867 val/balanced_acc=0.5000 val/precision=0.5867 val/recall=1.0000 val/specificity=0.0000 val/npv=0.0000 val/f1=0.7395 val/f1_macro=0.3697 val/mcc=0.0000 val/roc_auc=0.5659 val/pr_auc=0.6315 val/ece=0.0750 val
[single2d-

model.safetensors: reconstructing file:   0%|          |  0.00B /  115MB            

model.safetensors: downloading bytes:           |  0.00B            

[run] single2d-convnextv2_tiny-slabaip params=27.87M vram~0.8GB bs=2
[single2d-convnextv2_tiny-slabaip] params=27.87M lr=4.00e-04
[single2d-convnextv2_tiny-slabaip] ep01 loss=0.7714 val/acc=0.4133 val/balanced_acc=0.5000 val/precision=0.0000 val/recall=0.0000 val/specificity=1.0000 val/npv=0.4133 val/f1=0.0000 val/f1_macro=0.2925 val/mcc=0.0000 val/roc_auc=0.5372 val/pr_auc=0.6075 val/ece=0.1840 val
[single2d-convnextv2_tiny-slabaip] ep02 loss=0.7241 val/acc=0.5867 val/balanced_acc=0.5000 val/precision=0.5867 val/recall=1.0000 val/specificity=0.0000 val/npv=0.0000 val/f1=0.7395 val/f1_macro=0.3697 val/mcc=0.0000 val/roc_auc=0.5389 val/pr_auc=0.6191 val/ece=0.0455 val
[single2d-convnextv2_tiny-slabaip] ep03 loss=0.7030 val/acc=0.5867 val/balanced_acc=0.5000 val/precision=0.5867 val/recall=1.0000 val/specificity=0.0000 val/npv=0.0000 val/f1=0.7395 val/f1_macro=0.3697 val/mcc=0.0000 val/roc_auc=0.4886 val/pr_auc=0.5822 val/ece=0.0367 val
[single2d-convnextv2_tiny-slabaip] ep04 loss=0.6955

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.3MB            

...

[run] single2d-deit3_small_patch16_224-slabaip params=21.68M vram~0.7GB bs=2


[single2d-deit3_small_patch16_224-slabaip] params=21.68M lr=4.00e-04
[single2d-deit3_small_patch16_224-slabaip] ep01 loss=0.7540 val/acc=0.4133 val/balanced_acc=0.5000 val/precision=0.0000 val/recall=0.0000 val/specificity=1.0000 val/npv=0.4133 val/f1=0.0000 val/f1_macro=0.2925 val/mcc=0.0000 val/roc_auc=0.6601 val/pr_auc=0.7310 val/ece=0.2163 val
[single2d-deit3_small_patch16_224-slabaip] ep02 loss=0.6963 val/acc=0.6333 val/balanced_acc=0.5672 val/precision=0.6231 val/recall=0.9489 val/specificity=0.1855 val/npv=0.7188 val/f1=0.7523 val/f1_macro=0.5236 val/mcc=0.2143 val/roc_auc=0.6625 val/pr_auc=0.7402 val/ece=0.1168 val
[single2d-deit3_small_patch16_224-slabaip] ep03 loss=0.7087 val/acc=0.5867 val/balanced_acc=0.5000 val/precision=0.5867 val/recall=1.0000 val/specificity=0.0000 val/npv=0.0000 val/f1=0.7395 val/f1_macro=0.3697 val/mcc=0.0000 val/roc_auc=0.6571 val/pr_auc=0.7381 val/ece=0.0509 val
[single2d-deit3_small_patch16_224-slabaip] ep04 loss=0.6909 val/acc=0.5833 val/balanced_

model.safetensors: reconstructing file:   0%|          |  0.00B /  116MB            

model.safetensors: downloading bytes:           |  0.00B            

[run] single2d-maxvit_tiny_rw_224-slabaip params=28.55M vram~0.8GB bs=2
[single2d-maxvit_tiny_rw_224-slabaip] params=28.55M lr=4.00e-04
[single2d-maxvit_tiny_rw_224-slabaip] ep01 loss=0.7222 val/acc=0.6667 val/balanced_acc=0.6349 val/precision=0.6792 val/recall=0.8182 val/specificity=0.4516 val/npv=0.6364 val/f1=0.7423 val/f1_macro=0.6353 val/mcc=0.2918 val/roc_auc=0.7127 val/pr_auc=0.7637 val/ece=0.0596 val
[single2d-maxvit_tiny_rw_224-slabaip] ep02 loss=0.6791 val/acc=0.5967 val/balanced_acc=0.5121 val/precision=0.5926 val/recall=1.0000 val/specificity=0.0242 val/npv=1.0000 val/f1=0.7442 val/f1_macro=0.3957 val/mcc=0.1197 val/roc_auc=0.6727 val/pr_auc=0.6944 val/ece=0.1221 val
[single2d-maxvit_tiny_rw_224-slabaip] ep03 loss=0.6710 val/acc=0.6433 val/balanced_acc=0.6460 val/precision=0.7255 val/recall=0.6307 val/specificity=0.6613 val/npv=0.5578 val/f1=0.6748 val/f1_macro=0.6400 val/mcc=0.2876 val/roc_auc=0.7407 val/pr_auc=0.8122 val/ece=0.1046 val
[single2d-maxvit_tiny_rw_224-slabaip

In [29]:
def collate_dict(batch):
    items = {}
    for b in batch:
        for k, v in b.items():
            items.setdefault(k, []).append(v)
    out = {}
    for k, vs in items.items():
        if k == "idx":
            out[k] = torch.tensor([int(x) for x in vs])
        else:
            out[k] = torch.stack([torch.as_tensor(v) for v in vs])
    return out
_n = 0
if "_LOADER_CACHE" in globals():
    for _v in list(_LOADER_CACHE.values()):
        for _dl in _v:
            _dl.collate_fn = collate_dict
            _n += 1
print(f"[fix] collate_dict handles int idx; updated {_n} cached loaders. Re-run the sweep cell.")

[fix] collate_dict handles int idx; updated 3 cached loaders. Re-run the sweep cell.


In [ ]:
if SMOKE:
    print("===== SMOKE (synthetic, CPU, all code paths) =====")
    set_seed()
    CFG["num_workers"] = 0
    tr, va, te = build_smoke_loaders()
    b0 = next(iter(tr))
    print("batch x", tuple(b0["x"].shape), "v", tuple(b0["v"].shape), "y", tuple(b0["labels"].shape))
    smoke_specs = [
        {"name": "smoke-cnn3d", "fusion": "single3d", "enc3d": {"kind": "cnn3d"}},
        {"name": "smoke-convnext3d", "fusion": "single3d", "enc3d": {"kind": "convnext3d"}},
        {"name": "smoke-segresnet", "fusion": "single3d", "enc3d": {"kind": "segresnet", "params": {"init_filters": 8}}},
        {"name": "smoke-vit3d", "fusion": "single3d", "enc3d": {"kind": "vit3d", "params": {"dim": 64, "depth": 2, "patch": 16}}},
        {"name": "smoke-mednet10", "fusion": "single3d", "enc3d": {"kind": "mednet10", "params": {"pretrained": False, "inplanes": (16, 32, 64, 128)}}},
        {"name": "smoke-single2d", "fusion": "single2d", "view": 1, "enc2d": {"kind": "tiny2d", "params": {"features": (8, 16, 32, 64), "blocks": (1, 1, 1, 1)}}},
        {"name": "smoke-fusion-attn", "fusion": "attn",
         "enc3d": {"kind": "cnn3d", "params": {"features": (8, 16, 32, 64)}},
         "enc2d": {"kind": "tiny2d", "params": {"features": (8, 16, 32, 64), "blocks": (1, 1, 1, 1)}}},
        {"name": "smoke-fusion-crossgate", "fusion": "crossgate",
         "enc3d": {"kind": "cnn3d", "params": {"features": (8, 16, 32, 64)}},
         "enc2d": {"kind": "tiny2d", "params": {"features": (8, 16, 32, 64), "blocks": (1, 1, 1, 1)}}},
        {"name": "smoke-fusion-mamba", "fusion": "mamba",
         "enc3d": {"kind": "cnn3d", "params": {"features": (8, 16, 32, 64)}},
         "enc2d": {"kind": "tiny2d", "params": {"features": (8, 16, 32, 64), "blocks": (1, 1, 1, 1)}}},
        {"name": "smoke-fusion-film", "fusion": "film",
         "enc3d": {"kind": "cnn3d", "params": {"features": (8, 16, 32, 64)}},
         "enc2d": {"kind": "tiny2d", "params": {"features": (8, 16, 32, 64), "blocks": (1, 1, 1, 1)}}},
    ]
    for sp in smoke_specs:
        model = build_model(CFG, sp).to(DEVICE)
        npar = count_params(model)
        row, _hist = train_one(CFG, sp, (tr, va, te), os.path.join(SAVE_DIR, "smoke_ckpts"), wb=None)
        assert row["status"] == "ok", sp["name"]
        print(f"  OK {sp['name']:22s} pars={npar:6.2f}M test_acc={row['test']['acc']:.3f}")
    print("SMOKE_MODELS_OK")


## 9. X-AI on the winner + LIME on the best 2D single-view model + full reporting

- Runs the explainability pipeline (Grad-CAM 3D/2D, occlusion, integrated gradients, branch
  importance, fusion attention) on the sweep winner, using held-out test samples (balanced by class).
- Runs superpixel-LIME on the best single-2D model for a view-level explanation.
- Writes a Markdown report + all PNGs to `SAVE_DIR/xai/<run>` and copies them to Drive; also renders
  the full result table as CSV + PNG into `SAVE_DIR/report/` (mirrored on Drive) and logs to wandb.


In [ ]:
def flatten_rows_csv(rows, path, ci=True, n_boot=400):
    import csv as _csv
    keys = ["name", "fusion", "enc3d", "enc2d", "res3d", "res2d", "params", "best_ep", "sec_ep", "status"]
    mkeys = ["acc", "balanced_acc", "precision", "recall", "specificity", "npv", "f1", "f1_macro", "mcc",
             "roc_auc", "pr_auc", "ece", "logloss", "brier", "kappa", "youden", "tn", "fp", "fn", "tp", "n"]
    rows_by_name = {}
    for r in rows:
        if r.get("status") == "ok" and r.get("test"):
            rows_by_name[r["name"]] = r
    ci_cache = {}
    if ci:
        ck_dir = os.path.join(SAVE_DIR, "ckpts")
        for nm, r in rows_by_name.items():
            yp = os.path.join(ck_dir, f"{nm}_test_y.npy")
            pp = os.path.join(ck_dir, f"{nm}_test_p.npy")
            if os.path.isfile(yp) and os.path.isfile(pp):
                yt = np.load(yp); pt = np.load(pp)
                ci_cache[nm] = (bootstrap_ci(yt, pt, "roc_auc", n_boot=n_boot),
                                bootstrap_ci(yt, pt, "acc", n_boot=n_boot))
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = _csv.writer(fh)
        w.writerow(keys + ["test_" + k for k in mkeys]
                   + (["auc_ci_lo", "auc_ci_hi", "acc_ci_lo", "acc_ci_hi"] if ci else []))
        for r in rows:
            te = r.get("test") or {}
            s = r.get("spec") or {}
            auc_ci, acc_ci = ci_cache.get(r.get("name"), ((float("nan"), float("nan")), (float("nan"), float("nan"))))
            row = [r.get("name"), s.get("fusion"), (s.get("enc3d") or {}).get("kind"),
                   (s.get("enc2d") or {}).get("name") or (s.get("enc2d") or {}).get("kind"),
                   s.get("res3d"), s.get("res2d"), r.get("params"), r.get("best_ep"),
                   r.get("sec_ep"), r.get("status")]
            row += [te.get(k, "") for k in mkeys]
            if ci:
                row += [auc_ci[0], auc_ci[1], acc_ci[0], acc_ci[1]]
            w.writerow(row)

def copy_dir(src, dst):
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)
    print(f"[drive] copied {src} -> {dst}")

def write_survey_markdown(rows, out):
    by_name = {r.get("name"): r for r in rows}
    def rowa(spec, tier):
        r = by_name.get(spec["name"])
        te = (r or {}).get("test") or {}
        return r, te
    lines = ["# Survey coverage matrix", "",
             "Covers: 3D backbones (conv next-gen / MONAI seg-encoders / medical-net ResNet / ViT-3D), "
             "2D backbones (from-scratch + ImageNet-pretrained timm), fusion ops "
             "(concat/add/mul/self-attn/cross-attn+gate/Mamba-SSM/FiLM-SE), per-view 2D ablations, "
             "single-3D baselines, X-AI + full metrics + Drive sync.", ""]
    tiers = [("TIER A - single 3D branch", "A", "TIER_A_3D_SINGLE"),
             ("TIER B - single 2D view branch", "B", "TIER_B_2D_SINGLE"),
             ("TIER C - multiview fusion (3D + 3x2D)", "C", "TIER_C_FUSION")]
    for label, tier, varname in tiers:
        lines.append(f"## {label}")
        lines.append("| run | fusion | enc3d | enc2d | res3d | enabled | status | test-AUC |")
        lines.append("|---|---|---|---|---|---|---|---|")
        for spec in globals().get(varname, []):
            r = by_name.get(spec["name"])
            te = (r or {}).get("test") or {}
            status = "OK" if r and r.get("status") == "ok" else ("ERR" if r else "not run")
            en = "yes" if spec.get("enabled", True) else "no"
            lines.append(f"| {spec['name']} | {spec.get('fusion')} | "
                         f"{(spec.get('enc3d') or {}).get('kind','-')} | "
                         f"{(spec.get('enc2d') or {}).get('name','-') or (spec.get('enc2d') or {}).get('kind','-')} | "
                         f"{spec.get('res3d')} | {en} | {status} | {te.get('roc_auc', '')} |")
        lines.append("")
    ok = [r for r in rows if r.get("status") == "ok" and r.get("test")]
    if ok:
        b3 = best_of(ok, lambda s: s.get("spec", {}).get("fusion") == "single3d")
        b2 = best_of(ok, lambda s: s.get("spec", {}).get("fusion") == "single2d")
        bf = best_of(ok, lambda s: s.get("spec", {}).get("fusion") not in ("single3d", "single2d"))
        lines += ["## Interpretability guard", "",
                  "- best single-3D baseline: " + (fmt2(b3) if b3 else "n/a"),
                  "- best single-2D baseline: " + (fmt2(b2) if b2 else "n/a"),
                  "- best fusion:            " + (fmt2(bf) if bf else "n/a")]
        if bf:
            refs = [r for r in (b3, b2) if r is not None]
            best_single = max((r["test"]["roc_auc"] for r in refs), default=float("nan"))
            lines.append(f"- fusion gain over best single-stream: {bf['test']['roc_auc'] - best_single:+.4f} AUC "
                         "(positive = fusion adds value)")
    with open(out, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines))
    return "\n".join(lines)

def best_of(ok, pred):
    cand = [r for r in ok if pred(r)]
    return sorted(cand, key=lambda r: -(r["test"].get("roc_auc") or 0.0))[0] if cand else None

def fmt2(r):
    return f"{r['name']} AUC={r['test'].get('roc_auc'):.4f} acc={r['test'].get('acc'):.4f} f1={r['test'].get('f1'):.4f}"

def save_and_sync_report(rows, winner_name=None):
    report_dir = os.path.join(SAVE_DIR, "report")
    os.makedirs(report_dir, exist_ok=True)
    csvp = os.path.join(report_dir, "all_results.csv")
    flatten_rows_csv(rows, csvp)
    jdump(to_jsonable(rows), os.path.join(report_dir, "all_results.json"))
    ok_rows = [r for r in rows if r.get("status") == "ok" and r.get("test")]
    ok_rows = sorted(ok_rows, key=lambda r: -(r["test"].get("roc_auc") or 0.0))
    if ok_rows:
        plot_metric_table([{"name": r["name"], "params": round(r.get("params", 0), 2),
                            "val_acc": round((r.get("val") or {}).get("roc_auc") or r["test"].get("acc"), 4),
                            "val_auc": round((r.get("test") or {}).get("roc_auc") or 0, 4),
                            "test_acc": round(r["test"].get("acc", 0), 4),
                            "test_f1": round(r["test"].get("f1", 0), 4),
                            "test_auc": round(r["test"].get("roc_auc", 0), 4),
                            "test_mcc": round(r["test"].get("mcc", 0), 4),
                            "status": r["status"]} for r in ok_rows],
                          os.path.join(report_dir, "leaderboard.png"))
    jdump({"winner": winner_name, "n_rows": len(rows), "n_ok": len(ok_rows)}, os.path.join(report_dir, "summary.json"))
    smd = write_survey_markdown(rows, os.path.join(report_dir, "survey_matrix.md"))
    print(smd)
    for f in os.listdir(report_dir):
        copy_to_drive(os.path.join(report_dir, f), sub="report")
    return report_dir


In [ ]:
if SMOKE:
    print("===== SMOKE XAI paths =====")
    sp = {"name": "smoke-xai", "fusion": "attn",
          "enc3d": {"kind": "cnn3d", "params": {"features": (8, 16, 32, 64)}},
          "enc2d": {"kind": "tiny2d", "params": {"features": (8, 16, 32, 64), "blocks": (1, 1, 1, 1)}}}
    model = build_model(CFG, sp).to(DEVICE)
    model.eval()
    b = next(iter(te))
    x = b["x"].to(DEVICE).float(); v = b["v"].to(DEVICE).float(); y = int(b["labels"][0])
    odir = os.path.join(XAI_DIR, "smoke_xai")
    os.makedirs(odir, exist_ok=True)
    opts = {"gradcam3d": True, "gradcam2d": True, "occlusion": False, "ig2d": False}
    txt = explain_sample(model, CFG, x, v, y, "s0", odir, opts)
    names, drops, base = branch_importance(model, x, v, y)
    aw = fusion_attention_weights(model)
    print("branch_importance:", dict(zip(names, [round(float(d), 4) for d in drops])))
    print("fusion_attention:", aw)
    assert len(names) == 4 and aw is not None
    print("SMOKE_XAI_OK")


In [ ]:
if not SMOKE and RUN_XAI:
    rows = load_rows()
    if WINNER is None:
        WINNER = choose_winner(rows)
    if WINNER is None:
        print("[xai] no winner with test metrics - skipping X-AI")
    else:
        xai_win = WINNER["name"]
        print("\n===== X-AI on winner:", xai_win, "=====")
        xai_opts = {"gradcam3d": True, "gradcam2d": True, "occlusion": True,
                    "occlusion_grid": 3, "ig2d": True, "branch_ablation": True, "lime": True}
        try:
            out_dir, m = run_xai_pipeline(CFG, WINNER, xai_opts)
            xai_wb = init_wandb("xai-" + xai_win, config={"mode": "xai", "winner": xai_win})
            if xai_wb is not None:
                try:
                    xai_wb.log({k: v for k, v in m.items() if isinstance(v, (int, float))})
                    for f in sorted(os.listdir(out_dir)):
                        if f.endswith(".png"):
                            xai_wb.log({"xai": __import__("wandb").Image(os.path.join(out_dir, f))})
                except Exception as e:
                    print("[wandb-xai] log fail", e)
                finally:
                    try:
                        xai_wb.finish()
                    except Exception:
                        pass
            if os.path.exists(DRIVE_ROOT):
                copy_dir(out_dir, os.path.join(DRIVE_DIR, "xai", xai_win))
        except Exception as e:
            import traceback
            traceback.print_exc()
            print("[xai] FAILED:", type(e).__name__, str(e)[:400])
        single = best_single2d(rows)
        if single is not None and single["name"] != xai_win:
            print("\n===== LIME on best single-2D model:", single["name"], "=====")
            try:
                model2, spec2 = rebuild_model_from_row(CFG, single)
                _tr, _va, _te = get_loaders(CFG, spec2)
                ds2 = _te.dataset
                for ci, i in enumerate(pick_test_samples(_te, 1)):
                    x2, v2, y2 = sample_tensor(ds2, i)
                    view = int(spec2.get("view", 1))
                    imp, marked = lime_2d(model2, v2, view, int(y2), n_perturb=120, n_feat=14)
                    od2 = os.path.join(XAI_DIR, single["name"])
                    os.makedirs(od2, exist_ok=True)
                    fig, ax = plt.subplots(1, 2, figsize=(9, 4.4))
                    vi = _numpy(v2[0, view])
                    p1, p99 = np.percentile(vi, 1), np.percentile(vi, 99)
                    ax[0].imshow(marked); ax[0].set_title("superpixels"); ax[0].axis("off")
                    ax[1].imshow(vi, cmap="gray", vmin=p1, vmax=p99)
                    ax[1].imshow(imp, cmap="RdBu_r", alpha=0.5)
                    ax[1].set_title(f"LIME {VIEWS[view]}"); ax[1].axis("off")
                    fig.suptitle(f"{single['name']} | LIME view-level | true={int(y2)}")
                    fig.tight_layout()
                    lp = os.path.join(od2, f"lime_view{view}_sample{i}.png")
                    fig.savefig(lp, dpi=130); plt.close(fig)
                    print(f"  LIME sample idx={i} view={VIEWS[view]} saved")
                if os.path.exists(DRIVE_ROOT):
                    copy_dir(od2, os.path.join(DRIVE_DIR, "xai", single["name"]))
            except Exception as e:
                import traceback
                traceback.print_exc()
                print("[lime] FAILED:", type(e).__name__, str(e)[:300])
    print("\n===== FINAL REPORT & DRIVE SYNC =====")
    report_dir = save_and_sync_report(rows, (WINNER or {}).get("name"))
    if WINNER:
        ck = os.path.join(SAVE_DIR, "ckpts", WINNER["name"] + ".pt")
        if os.path.isfile(ck):
            copy_to_drive(ck, sub="ckpts")
    print("saved report:", report_dir)
    print("drive dir:", DRIVE_DIR)


In [ ]:
if SMOKE:
    print("\nSMOKE_OK - every backbone/fusion/xai path ran on synthetic data.")
if not IN_COLAB and not SMOKE and not RUN_SWEEP and not RUN_XAI:
    print("Heavy cells gated. Set MVX_SWEEP=1 / MVX_XAI=1 or run on Colab GPU.")


## How to run (Colab GPU)

1. **Runtime**: GPU (A100/H100 recommended). Execute cells top to bottom; cell 2 installs
   `monai / timm / sklearn / skimage / shap / wandb`; cell 3 mounts Drive and reads `HF_TOKEN` +
   `WANDB_API_KEY` from Colab Secrets.
2. **Data**: cell in section 2 streams Harvard-GF (per-scan `.npz`) to consolidated raw **200^3**
   arrays on `/content/glaucoma_hf_200` (once, ~26 GB disk; mmap after) and caches the 3 en-face
   views per split. Re-runs reuse the cache.
3. **Tune scope** (top of section 2 `CFG` + section 6): `epochs`, `batch_size`, `grad_accum`;
   per-spec `enabled`, `res3d`, `res2d`, `epochs`, `batch_size`. Env overrides:
   `MVX_SWEEP=0/1`, `MVX_XAI=0/1`, `MVX_TIER_A/B/C`, `MVX_SMOKE=1`.
4. **Budget**: conv-3D runs keep the **raw 200^3** (10-25 min/run); transformer rows run at 160
   (res % 32 = 0) ~10-30 min/run. Each finished model immediately exports ROC/PR/calibration + confusion
   + val-history PNGs to `Drive/.../multiview_sota_sweep/figures/<run>/` and logs them to wandb, so you
   can watch the leaderboard grow while it runs. Reduce `epochs` of heavy rows to 6 and disable
   transformer rows for a quick pass. Resume-safe: interrupted cell keeps finished rows.
5. **Cheap validation first**: `MVX_SMOKE=1` runs synthetic mini training of every backbone/fusion +
   a short X-AI path on CPU — do this before spending GPU hours.
6. **After the sweep**: the X-AI cell (section 9) explains the winner (Grad-CAM 3D/2D, occlusion,
   integrated gradients, fusion attention, branch importance), runs LIME on the best single-2D model,
   and saves metrics CSV/PNG + figures + this report under Drive
   `MyDrive/MasterBKDN/Thesis/multiview_sota_sweep/`.

## Guardrails inherited from the repo

- Raw data is never downsampled on disk (`STORE_RES=200`; resizing happens on-the-fly per backbone).
- Loss = CrossEntropy on raw logits (2-class), AMP bf16/fp16 + GradScaler, grad-accum, early stop,
  eval on val each epoch + full test metrics at the best epoch.
- BatchNorm is avoided for tiny batches (GroupNorm/LayerNorm in from-scratch nets); MONAI rows keep
  their native norms but train at bs>=2.
